In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import h5py
import pandas as pd  # 添加这行
import scipy.io
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import os
import json
from copy import deepcopy
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap

# TabNet
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.metrics import Metric

# 设置随机种子
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 设置设备和路径
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ 使用设备: {device}")

export_path = './tabnet_simplified_experiment/'
os.makedirs(export_path, exist_ok=True)
os.makedirs(os.path.join(export_path, 'visualizations'), exist_ok=True)
os.makedirs(os.path.join(export_path, 'models'), exist_ok=True)
os.makedirs(os.path.join(export_path, 'logs'), exist_ok=True)

print("✅ 环境设置完成")

# 📋 5层级TabNet配置 + 精心调优的超参数
TABNET_COMPLETE_CONFIGS = {
    'micro_config': {
        # 模型架构
        'n_d': 16,
        'n_a': 16,
        'n_steps': 2,
        'gamma': 1.8,
        'n_independent': 1,
        'n_shared': 1,
        'lambda_sparse': 8e-3,
        'mask_type': 'sparsemax',
        
        # 优化器参数
        'learning_rate': 8e-3,
        'weight_decay': 5e-4,
        'momentum': 0.4,
        'clip_value': 2.5,
        
        # 训练参数
        'batch_size': 256,
        'virtual_batch_size_ratio': 0.4,
        'max_epochs': 150,
        'patience': 25,
        
        # 学习率调度
        'scheduler_patience': 8,
        'scheduler_factor': 0.7,
        
        'estimated_params': '~1M'
    },
    
    'small_config': {
        # 模型架构
        'n_d': 32,
        'n_a': 32,
        'n_steps': 3,
        'gamma': 1.6,
        'n_independent': 2,
        'n_shared': 2,
        'lambda_sparse': 3e-3,
        'mask_type': 'sparsemax',
        
        # 优化器参数
        'learning_rate': 5e-3,
        'weight_decay': 3e-4,
        'momentum': 0.35,
        'clip_value': 2.0,
        
        # 训练参数
        'batch_size': 512,
        'virtual_batch_size_ratio': 0.35,
        'max_epochs': 180,
        'patience': 30,
        
        # 学习率调度
        'scheduler_patience': 10,
        'scheduler_factor': 0.6,
        
        'estimated_params': '~3M'
    },
    
    'medium_config': {
        # 模型架构
        'n_d': 64,
        'n_a': 64,
        'n_steps': 4,
        'gamma': 1.4,
        'n_independent': 3,
        'n_shared': 2,
        'lambda_sparse': 8e-4,
        'mask_type': 'sparsemax',
        
        # 优化器参数
        'learning_rate': 3e-3,
        'weight_decay': 2e-4,
        'momentum': 0.3,
        'clip_value': 1.8,
        
        # 训练参数
        'batch_size': 512,
        'virtual_batch_size_ratio': 0.3,
        'max_epochs': 200,
        'patience': 35,
        
        # 学习率调度
        'scheduler_patience': 12,
        'scheduler_factor': 0.5,
        
        'estimated_params': '~8M'
    },
    
    'large_config': {
        # 模型架构
        'n_d': 128,
        'n_a': 128,
        'n_steps': 5,
        'gamma': 1.25,
        'n_independent': 4,
        'n_shared': 3,
        'lambda_sparse': 3e-4,
        'mask_type': 'sparsemax',
        
        # 优化器参数
        'learning_rate': 2e-3,
        'weight_decay': 1e-4,
        'momentum': 0.25,
        'clip_value': 1.5,
        
        # 训练参数
        'batch_size': 1024,
        'virtual_batch_size_ratio': 0.25,
        'max_epochs': 150,
        'patience': 40,
        
        # 学习率调度
        'scheduler_patience': 15,
        'scheduler_factor': 0.4,
        
        'estimated_params': '~20M'
    },
    
    'xlarge_config': {
        # 模型架构
        'n_d': 256,
        'n_a': 256,
        'n_steps': 6,
        'gamma': 1.15,
        'n_independent': 5,
        'n_shared': 3,
        'lambda_sparse': 1e-4,
        'mask_type': 'sparsemax',
        
        # 优化器参数
        'learning_rate': 1.5e-3,
        'weight_decay': 8e-5,
        'momentum': 0.2,
        'clip_value': 1.2,
        
        # 训练参数
        'batch_size': 1024,
        'virtual_batch_size_ratio': 0.2,
        'max_epochs': 150,
        'patience': 45,
        
        # 学习率调度
        'scheduler_patience': 18,
        'scheduler_factor': 0.3,
        
        'estimated_params': '~35M (对标4x4096全连接)'
    }
}

print("📋 TabNet完整配置定义完成:")
for config_name, config in TABNET_COMPLETE_CONFIGS.items():
    print(f"  {config_name}: {config['estimated_params']}, LR={config['learning_rate']:.2e}")

# 📂 数据加载和预处理（保持原有逻辑）
print("📂 加载数据...")
f = h5py.File('/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38_no_label43.mat','r')
arrays = {}
for k, v in f.items():
    arrays[k] = np.array(v)
f.close()

train_data = arrays['data'].transpose()
train_region = arrays['region'].transpose()
prob_idx = arrays['prob_idx'].transpose()
print(f"原始数据形状: {train_data.shape}")
print(f"原始标签形状: {train_region.shape}")
print(f"prob_idx形状: {prob_idx.shape}")

del arrays, f

# 创建数据分割
test_indices = np.where(prob_idx == 38)[0]
train_val_indices = np.where(prob_idx != 38)[0]

test_data = train_data[test_indices, :]
test_labels = train_region[test_indices, :]
train_val_data = train_data[train_val_indices, :]
train_val_labels = train_region[train_val_indices, :]

print(f"测试集形状: {test_data.shape}")
print(f"训练+验证集形状: {train_val_data.shape}")

# 进一步分割训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(
    train_val_data, train_val_labels, 
    test_size=0.2, random_state=42, stratify=np.argmax(train_val_labels, axis=1)
)

print(f"最终训练集形状: {X_train.shape}")
print(f"最终验证集形状: {X_val.shape}")
print(f"最终测试集形状: {test_data.shape}")

del train_data, train_region, prob_idx, train_val_data, train_val_labels, test_indices, train_val_indices

# 标准化（只在训练集上拟合）
print("⚙️ 应用标准化...")
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(test_data)

# 转换标签格式 (从one-hot到整数标签，TabNet需要)
y_train_int = np.argmax(y_train, axis=1)
y_val_int = np.argmax(y_val, axis=1)
y_test_int = np.argmax(test_labels, axis=1)

print("✅ 数据加载完成，变量确认:")
print(f"  训练集: {X_train_scaled.shape}, 标签: {y_train_int.shape}")
print(f"  验证集: {X_val_scaled.shape}, 标签: {y_val_int.shape}")  
print(f"  测试集: {X_test_scaled.shape}, 标签: {y_test_int.shape}")
print(f"  特征维度: {X_train_scaled.shape[1]}")
print(f"  类别数量: {len(np.unique(y_train_int))}")

# 🔧 训练监控系统
class TabNetTrainingMonitor:
    """完整的TabNet训练监控系统"""
    
    def __init__(self, config_name):
        self.config_name = config_name
        self.history = {
            'epoch': [],
            'train_f1_macro': [],
            'val_f1_macro': [], 
            'test_f1_macro': [],
            'train_loss': [],
            'val_loss': [],
            'test_loss': [],
            'train_accuracy': [],
            'val_accuracy': [],
            'test_accuracy': [],
            'learning_rate': [],
            'overfitting_gap': [],
            'generalization_gap': []
        }
        self.feature_importance_history = []
        
    def calculate_cross_entropy_loss(self, y_true, y_pred_proba):
        """计算交叉熵损失"""
        # 避免log(0)
        y_pred_proba = np.clip(y_pred_proba, 1e-15, 1 - 1e-15)
        return -np.mean(np.log(y_pred_proba[np.arange(len(y_true)), y_true]))
        
    def on_epoch_end(self, epoch, model, X_train, y_train, X_val, y_val, X_test, y_test, 
                     current_lr=None):
        """每个epoch结束后的完整评估"""
        
        try:
            # 1. 三个数据集的预测
            print(f"     Epoch {epoch+1} - 评估三个数据集...")
            
            train_preds = model.predict(X_train)
            train_proba = model.predict_proba(X_train)
            
            val_preds = model.predict(X_val)
            val_proba = model.predict_proba(X_val)
            
            test_preds = model.predict(X_test)
            test_proba = model.predict_proba(X_test)
            
            # 2. 计算macro F1和准确率
            train_f1 = f1_score(y_train, train_preds, average='macro')
            val_f1 = f1_score(y_val, val_preds, average='macro')
            test_f1 = f1_score(y_test, test_preds, average='macro')
            
            train_acc = accuracy_score(y_train, train_preds)
            val_acc = accuracy_score(y_val, val_preds)
            test_acc = accuracy_score(y_test, test_preds)
            
            # 3. 计算损失
            train_loss = self.calculate_cross_entropy_loss(y_train, train_proba)
            val_loss = self.calculate_cross_entropy_loss(y_val, val_proba)
            test_loss = self.calculate_cross_entropy_loss(y_test, test_proba)
            
            # 4. 计算过拟合和泛化指标
            overfitting_gap = train_f1 - val_f1
            generalization_gap = val_f1 - test_f1
            
            # 5. 记录历史
            self.history['epoch'].append(epoch + 1)
            self.history['train_f1_macro'].append(train_f1)
            self.history['val_f1_macro'].append(val_f1)
            self.history['test_f1_macro'].append(test_f1)
            self.history['train_accuracy'].append(train_acc)
            self.history['val_accuracy'].append(val_acc)
            self.history['test_accuracy'].append(test_acc)
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['test_loss'].append(test_loss)
            self.history['overfitting_gap'].append(overfitting_gap)
            self.history['generalization_gap'].append(generalization_gap)
            
            if current_lr is not None:
                self.history['learning_rate'].append(current_lr)
            
            # 6. 特征重要性（每10个epoch记录一次）
            if epoch % 10 == 0:
                try:
                    feature_imp = model.feature_importances_
                    self.feature_importance_history.append({
                        'epoch': epoch + 1,
                        'importance': feature_imp.copy()
                    })
                except:
                    pass
            
            # 7. 打印当前epoch统计
            print(f"      F1    - Train: {train_f1:.4f}, Val: {val_f1:.4f}, Test: {test_f1:.4f}")
            print(f"      Loss  - Train: {train_loss:.4f}, Val: {val_loss:.4f}, Test: {test_loss:.4f}")
            print(f"      Gap   - Overfit: {overfitting_gap:+.4f}, General: {generalization_gap:+.4f}")
            
            return val_f1  # 返回验证F1用于early stopping
            
        except Exception as e:
            print(f"    ⚠️ 评估失败: {e}")
            return 0.0
    
    def plot_training_summary(self, save_path=None):
        """绘制训练总结"""
        
        if len(self.history['epoch']) < 2:
            return
            
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        
        epochs = self.history['epoch']
        
        # 1. F1 Score曲线
        axes[0,0].plot(epochs, self.history['train_f1_macro'], 'b-', label='Train F1', linewidth=2)
        axes[0,0].plot(epochs, self.history['val_f1_macro'], 'g-', label='Val F1', linewidth=2)
        axes[0,0].plot(epochs, self.history['test_f1_macro'], 'r--', label='Test F1 (观察)', linewidth=2, alpha=0.7)
        axes[0,0].set_title(f'{self.config_name} - F1 Score Progress')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].set_ylabel('Macro F1 Score')
        axes[0,0].legend()
        axes[0,0].grid(True, alpha=0.3)
        
        # 2. Loss曲线
        axes[0,1].plot(epochs, self.history['train_loss'], 'b-', label='Train Loss', linewidth=2)
        axes[0,1].plot(epochs, self.history['val_loss'], 'g-', label='Val Loss', linewidth=2)
        axes[0,1].plot(epochs, self.history['test_loss'], 'r--', label='Test Loss (观察)', linewidth=2, alpha=0.7)
        axes[0,1].set_title(f'{self.config_name} - Loss Progress')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].set_ylabel('Cross Entropy Loss')
        axes[0,1].legend()
        axes[0,1].grid(True, alpha=0.3)
        
        # 3. 准确率曲线
        axes[0,2].plot(epochs, self.history['train_accuracy'], 'b-', label='Train Acc', linewidth=2)
        axes[0,2].plot(epochs, self.history['val_accuracy'], 'g-', label='Val Acc', linewidth=2)
        axes[0,2].plot(epochs, self.history['test_accuracy'], 'r--', label='Test Acc (观察)', linewidth=2, alpha=0.7)
        axes[0,2].set_title(f'{self.config_name} - Accuracy Progress')
        axes[0,2].set_xlabel('Epoch')
        axes[0,2].set_ylabel('Accuracy')
        axes[0,2].legend()
        axes[0,2].grid(True, alpha=0.3)
        
        # 4. 过拟合分析
        axes[1,0].plot(epochs, self.history['overfitting_gap'], 'orange', linewidth=2, label='Train-Val Gap')
        axes[1,0].axhline(y=0.05, color='red', linestyle='--', alpha=0.7, label='Warning Threshold')
        axes[1,0].axhline(y=0, color='black', linestyle='-', alpha=0.3)
        axes[1,0].set_title('Overfitting Analysis')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].set_ylabel('Train F1 - Val F1')
        axes[1,0].legend()
        axes[1,0].grid(True, alpha=0.3)
        
        # 5. 泛化分析
        axes[1,1].plot(epochs, self.history['generalization_gap'], 'purple', linewidth=2, label='Val-Test Gap')
        axes[1,1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
        axes[1,1].set_title('Generalization Analysis')
        axes[1,1].set_xlabel('Epoch')
        axes[1,1].set_ylabel('Val F1 - Test F1')
        axes[1,1].legend()
        axes[1,1].grid(True, alpha=0.3)
        
        # 6. 学习率曲线（如果有记录）
        if self.history['learning_rate']:
            axes[1,2].plot(epochs[:len(self.history['learning_rate'])], 
                          self.history['learning_rate'], 'brown', linewidth=2)
            axes[1,2].set_title('Learning Rate Schedule')
            axes[1,2].set_xlabel('Epoch')
            axes[1,2].set_ylabel('Learning Rate')
            axes[1,2].set_yscale('log')
        else:
            axes[1,2].text(0.5, 0.5, 'Learning Rate\nNot Tracked', 
                          ha='center', va='center', transform=axes[1,2].transAxes,
                          fontsize=12, bbox=dict(boxstyle='round', facecolor='lightgray'))
        axes[1,2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            
        plt.show()
        
        # 打印最佳性能
        if self.history['val_f1_macro']:
            best_val_epoch = np.argmax(self.history['val_f1_macro'])
            best_val_f1 = self.history['val_f1_macro'][best_val_epoch]
            corresponding_test_f1 = self.history['test_f1_macro'][best_val_epoch]
            
            print(f"\n🏆 {self.config_name} 最佳性能:")
            print(f"   最佳验证F1: {best_val_f1:.4f} (Epoch {best_val_epoch + 1})")
            print(f"   对应测试F1: {corresponding_test_f1:.4f}")
            print(f"   验证-测试Gap: {best_val_f1 - corresponding_test_f1:+.4f}")




class CheckpointManager:
    """检查点管理器 - 保存模型和所有必要信息用于后续分析"""
    
    def __init__(self, config_name, base_path):
        self.config_name = config_name
        self.checkpoint_dir = os.path.join(base_path, 'checkpoints', config_name)
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        self.checkpoint_info = []
        
    def save_checkpoint(self, model, epoch, config, training_history, dataset_info):
        """保存完整检查点"""
        checkpoint_name = f"epoch_{epoch:04d}"
        checkpoint_path = os.path.join(self.checkpoint_dir, checkpoint_name)
        os.makedirs(checkpoint_path, exist_ok=True)
        
        # 1. 保存模型
        model_path = os.path.join(checkpoint_path, "model.zip")
        model.save_model(model_path)
        
        # 2. 保存配置
        config_path = os.path.join(checkpoint_path, "config.json")
        with open(config_path, 'w') as f:
            json.dump(config, f, indent=2)
        
        # 3. 保存训练历史
        history_path = os.path.join(checkpoint_path, "training_history.json")
        with open(history_path, 'w') as f:
            json.dump(training_history, f, indent=2)
        
        # 4. 保存数据集信息（用于验证）
        dataset_info_path = os.path.join(checkpoint_path, "dataset_info.json")
        with open(dataset_info_path, 'w') as f:
            json.dump(dataset_info, f, indent=2)
        
        # 5. 记录检查点信息
        checkpoint_record = {
            'epoch': epoch,
            'checkpoint_path': checkpoint_path,
            'model_path': model_path,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            'val_f1': training_history.get('val_f1_macro', [])[-1] if training_history.get('val_f1_macro') else None,
            'train_f1': training_history.get('train_f1_macro', [])[-1] if training_history.get('train_f1_macro') else None,
            'config_name': self.config_name
        }
        self.checkpoint_info.append(checkpoint_record)
        
        # 6. 保存检查点索引
        index_path = os.path.join(self.checkpoint_dir, "checkpoint_index.json")
        with open(index_path, 'w') as f:
            json.dump(self.checkpoint_info, f, indent=2)
        
        print(f"💾 检查点已保存: {checkpoint_path}")
        return checkpoint_path
        
        
# 🚦 Early Stopping系统
class ValidationEarlyStopping:
    """基于验证F1的Early Stopping"""
    
    def __init__(self, patience=35, min_delta=5e-5, restore_best_weights=False):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_val_f1 = -np.inf
        self.best_epoch = 0
        self.wait = 0
        self.stopped_epoch = 0
        self.should_stop = False
        
    def on_epoch_end(self, epoch, val_f1):
        """检查是否应该停止训练"""
        
        if val_f1 > self.best_val_f1 + self.min_delta:
            self.best_val_f1 = val_f1
            self.best_epoch = epoch
            self.wait = 0
            print(f"      ✅ 新的最佳验证F1: {val_f1:.4f}")
        else:
            self.wait += 1
            print(f"      ⏳ 验证F1未改善: {self.wait}/{self.patience}")
            
            if self.wait >= self.patience:
                self.stopped_epoch = epoch
                self.should_stop = True
                print(f"      🛑 Early Stopping 触发! 最佳验证F1: {self.best_val_f1:.4f} (Epoch {self.best_epoch + 1})")
                return True
        
        return False
        
    def get_best_info(self):
        """获取最佳模型信息"""
        return {
            'best_val_f1': self.best_val_f1,
            'best_epoch': self.best_epoch,
            'stopped_epoch': self.stopped_epoch if self.should_stop else None
        }

class CheckpointAnalyzer:
    """检查点分析器 - 后期分析所有保存的检查点"""
    
    def __init__(self, checkpoint_dir, X_train, y_train, X_val, y_val, X_test, y_test):
        self.checkpoint_dir = checkpoint_dir
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.X_test = X_test
        self.y_test = y_test
        
        # 加载检查点索引
        index_path = os.path.join(checkpoint_dir, "checkpoint_index.json")
        with open(index_path, 'r') as f:
            self.checkpoint_info = json.load(f)
        
        print(f"📂 找到 {len(self.checkpoint_info)} 个检查点")

    def analyze_all_checkpoints(self):
        """分析所有检查点的三数据集性能"""
        
        results = []
        
        print(f"\n🔍 开始分析所有检查点...")
        print("="*80)
        
        for checkpoint in tqdm(self.checkpoint_info, desc="分析检查点"):
            epoch = checkpoint['epoch']
            model_path = checkpoint['model_path']
            
            try:
                # 加载模型
                model = TabNetClassifier()
                model.load_model(model_path)
                
                # 预测三个数据集
                train_preds = model.predict(self.X_train)
                train_proba = model.predict_proba(self.X_train)
                
                val_preds = model.predict(self.X_val)
                val_proba = model.predict_proba(self.X_val)
                
                test_preds = model.predict(self.X_test)
                test_proba = model.predict_proba(self.X_test)
                
                # 计算指标
                result = {
                    'epoch': epoch,
                    'checkpoint_path': checkpoint['checkpoint_path'],
                    
                    # F1 Scores
                    'train_f1_macro': f1_score(self.y_train, train_preds, average='macro'),
                    'val_f1_macro': f1_score(self.y_val, val_preds, average='macro'),
                    'test_f1_macro': f1_score(self.y_test, test_preds, average='macro'),
                    
                    # Accuracy
                    'train_accuracy': accuracy_score(self.y_train, train_preds),
                    'val_accuracy': accuracy_score(self.y_val, val_preds),
                    'test_accuracy': accuracy_score(self.y_test, test_preds),
                    
                    # Loss
                    'train_loss': self.calculate_cross_entropy_loss(self.y_train, train_proba),
                    'val_loss': self.calculate_cross_entropy_loss(self.y_val, val_proba),
                    'test_loss': self.calculate_cross_entropy_loss(self.y_test, test_proba),
                }
                
                # 计算gaps
                result['overfitting_gap'] = result['train_f1_macro'] - result['val_f1_macro']
                result['generalization_gap'] = result['val_f1_macro'] - result['test_f1_macro']
                
                results.append(result)
                
            except Exception as e:
                print(f"\n  ❌ Epoch {epoch} 分析失败: {e}")
                continue
        
        # 创建DataFrame并保存
        if results:
            results_df = pd.DataFrame(results)
            
            # 保存分析结果
            results_path = os.path.join(self.checkpoint_dir, "checkpoint_analysis_results.csv")
            results_df.to_csv(results_path, index=False)
            print(f"\n💾 分析结果已保存: {results_path}")
            
            # 打印汇总
            print("\n📊 检查点分析结果汇总:")
            print("="*80)
            print(results_df[['epoch', 'train_f1_macro', 'val_f1_macro', 'test_f1_macro', 
                            'overfitting_gap', 'generalization_gap']].round(4).to_string(index=False))
            
            # 生成过拟合分析图
            self.plot_overfitting_analysis(results_df)
            
            return results_df
        else:
            print("❌ 没有成功分析的检查点")
            return pd.DataFrame()


    def plot_overfitting_analysis(self, results_df):
        """绘制详细的过拟合分析图"""
        
        fig, axes = plt.subplots(3, 2, figsize=(15, 12))
        
        epochs = results_df['epoch']
        
        # 1. F1 Score进展
        ax = axes[0, 0]
        ax.plot(epochs, results_df['train_f1_macro'], 'b-', label='Train F1', linewidth=2)
        ax.plot(epochs, results_df['val_f1_macro'], 'g-', label='Val F1', linewidth=2)
        ax.plot(epochs, results_df['test_f1_macro'], 'r--', label='Test F1', linewidth=2)
        ax.set_title('F1 Score Evolution')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Macro F1 Score')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 2. Loss进展
        ax = axes[0, 1]
        ax.plot(epochs, results_df['train_loss'], 'b-', label='Train Loss', linewidth=2)
        ax.plot(epochs, results_df['val_loss'], 'g-', label='Val Loss', linewidth=2)
        ax.plot(epochs, results_df['test_loss'], 'r--', label='Test Loss', linewidth=2)
        ax.set_title('Loss Evolution')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Cross Entropy Loss')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 3. 过拟合Gap
        ax = axes[1, 0]
        ax.plot(epochs, results_df['overfitting_gap'], 'orange', linewidth=2)
        ax.axhline(y=0.05, color='red', linestyle='--', alpha=0.7, label='Warning (5%)')
        ax.axhline(y=0.02, color='yellow', linestyle='--', alpha=0.7, label='Caution (2%)')
        ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        ax.set_title('Overfitting Gap (Train F1 - Val F1)')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Gap')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 4. 泛化Gap
        ax = axes[1, 1]
        ax.plot(epochs, results_df['generalization_gap'], 'purple', linewidth=2)
        ax.axhline(y=0.02, color='red', linestyle='--', alpha=0.7, label='Warning (±2%)')
        ax.axhline(y=-0.02, color='red', linestyle='--', alpha=0.7)
        ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        ax.set_title('Generalization Gap (Val F1 - Test F1)')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Gap')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 5. 准确率进展
        ax = axes[2, 0]
        ax.plot(epochs, results_df['train_accuracy'], 'b-', label='Train Acc', linewidth=2)
        ax.plot(epochs, results_df['val_accuracy'], 'g-', label='Val Acc', linewidth=2)
        ax.plot(epochs, results_df['test_accuracy'], 'r--', label='Test Acc', linewidth=2)
        ax.set_title('Accuracy Evolution')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 6. 最佳epoch标记
        ax = axes[2, 1]
        # 找到最佳验证F1的epoch
        best_val_idx = results_df['val_f1_macro'].idxmax()
        best_epoch = results_df.loc[best_val_idx, 'epoch']
        
        # 绘制验证F1曲线并标记最佳点
        ax.plot(epochs, results_df['val_f1_macro'], 'g-', linewidth=2)
        ax.scatter([best_epoch], [results_df.loc[best_val_idx, 'val_f1_macro']], 
                  color='red', s=100, zorder=5, label=f'Best (Epoch {best_epoch})')
        ax.set_title('Best Model Selection')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Validation F1 Score')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # 保存图表
        plot_path = os.path.join(self.checkpoint_dir, "overfitting_analysis.png")
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"📊 过拟合分析图已保存: {plot_path}")
        
        # 打印分析总结
        print(f"\n📋 过拟合分析总结:")
        print(f"  最佳验证F1: {results_df.loc[best_val_idx, 'val_f1_macro']:.4f} (Epoch {best_epoch})")
        print(f"  对应测试F1: {results_df.loc[best_val_idx, 'test_f1_macro']:.4f}")
        print(f"  最终过拟合Gap: {results_df.iloc[-1]['overfitting_gap']:+.4f}")
        print(f"  最大过拟合Gap: {results_df['overfitting_gap'].max():+.4f}")
        print(f"  开始过拟合的Epoch: {results_df[results_df['overfitting_gap'] > 0.05]['epoch'].min() if any(results_df['overfitting_gap'] > 0.05) else 'N/A'}")
    
    @staticmethod
    def calculate_cross_entropy_loss(y_true, y_pred_proba):
        """计算交叉熵损失"""
        y_pred_proba = np.clip(y_pred_proba, 1e-15, 1 - 1e-15)
        return -np.mean(np.log(y_pred_proba[np.arange(len(y_true)), y_true]))

        
class SimplifiedTabNetTrainer:
    """简化的TabNet训练器 - 使用预定义超参数"""
    
    def __init__(self, config_name, X_train, y_train, X_val, y_val, X_test, y_test):
        self.config_name = config_name
        self.config = TABNET_COMPLETE_CONFIGS[config_name]
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.X_test = X_test
        self.y_test = y_test
        
        # 初始化监控和早停
        self.monitor = TabNetTrainingMonitor(config_name)
        self.early_stopping = ValidationEarlyStopping(
            patience=self.config['patience'], 
            min_delta=5e-5
        )
        self.model = None
        self.training_completed = False
        
    def build_model(self):
        """构建模型"""
        
        print(f"🔧 构建 {self.config_name} 模型...")
        
        # 分离架构参数和训练参数
        tabnet_arch_params = {
            'n_d': self.config['n_d'],
            'n_a': self.config['n_a'],
            'n_steps': self.config['n_steps'],
            'gamma': self.config['gamma'],
            'n_independent': self.config['n_independent'],
            'n_shared': self.config['n_shared'],
            'lambda_sparse': self.config['lambda_sparse'],
            'mask_type': self.config['mask_type']
        }
        
        # 创建模型
        self.model = TabNetClassifier(
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(
                lr=self.config['learning_rate'],
                weight_decay=self.config['weight_decay']
            ),
            # 训练参数
            momentum=self.config['momentum'],
            clip_value=self.config['clip_value'],
            # 学习率调度器
            scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
            scheduler_params=dict(
                mode='max',
                factor=self.config['scheduler_factor'],
                patience=self.config['scheduler_patience'],
                verbose=True,
                min_lr=1e-6
            ),
            device_name='cuda' if torch.cuda.is_available() else 'cpu',
            verbose=1,
            # 架构参数
            **tabnet_arch_params
        )
        
        print(f"✅ {self.config_name} 模型构建完成")
        print(f"    架构参数: n_d={self.config['n_d']}, n_steps={self.config['n_steps']}, gamma={self.config['gamma']:.2f}")
        print(f"    优化参数: lr={self.config['learning_rate']:.2e}, wd={self.config['weight_decay']:.2e}")
        print(f"    预估参数: {self.config['estimated_params']}")
        


    def train_with_checkpoints(self, checkpoint_frequency=5):
        """使用定期检查点的训练 - 完整实现"""
        
        if self.model is None:
            self.build_model()
        
        print(f"\n🚀 开始 {self.config_name} 训练 (每{checkpoint_frequency}个epoch保存检查点)...")
        print(f"    最大轮数: {self.config['max_epochs']}")
        print(f"    Early stopping patience: {self.config['patience']}")
        print(f"    批次大小: {self.config['batch_size']}")
        
        # 初始化检查点管理器
        self.checkpoint_manager = CheckpointManager(self.config_name, export_path)
        
        # 保存数据集信息
        dataset_info = {
            'n_train': len(self.X_train),
            'n_val': len(self.X_val),
            'n_test': len(self.X_test),
            'n_features': self.X_train.shape[1],
            'n_classes': len(np.unique(self.y_train)),
            'feature_dim': self.X_train.shape[1],
            'train_samples': len(self.X_train),
            'val_samples': len(self.X_val),
            'test_samples': len(self.X_test)
        }
        
        # 计算批次参数
        virtual_batch_size = max(32, int(self.config['batch_size'] * self.config['virtual_batch_size_ratio']))
        print(f"    虚拟批次大小: {virtual_batch_size}")
        
        # 训练状态追踪
        start_time = time.time()
        best_val_f1 = -np.inf
        patience_counter = 0
        current_epoch = 0
        
        # 创建自定义F1 metric
        from pytorch_tabnet.metrics import Metric
        from sklearn.metrics import f1_score as sklearn_f1
        
        class MacroF1(Metric):
            def __init__(self):
                self._name = "macro_f1"
                self._maximize = True
            
            def __call__(self, y_true, y_score):
                y_pred = np.argmax(y_score, axis=1)
                return sklearn_f1(y_true, y_pred, average='macro')
        
        try:
            total_epochs = self.config['max_epochs']
            
            # 第一次训练 - 完整初始化模型
            print(f"\n🔥 开始初始训练...")
            
            while current_epoch < total_epochs:
                # 计算本轮训练的epochs数
                epochs_this_round = min(checkpoint_frequency, total_epochs - current_epoch)
                end_epoch = current_epoch + epochs_this_round
                
                print(f"\n📍 训练轮次: Epochs {current_epoch + 1}-{end_epoch}")
                
                if current_epoch == 0:
                    # 第一轮：正常训练
                    print("    首次训练，初始化模型...")
                    
                    self.model.fit(
                        X_train=self.X_train,
                        y_train=self.y_train,
                        eval_set=[(self.X_val, self.y_val)],
                        eval_name=['val'],
                        eval_metric=['accuracy', 'logloss', MacroF1],
                        max_epochs=epochs_this_round,
                        patience=0,  # 不使用内置的early stopping
                        batch_size=self.config['batch_size'],
                        virtual_batch_size=virtual_batch_size,
                        num_workers=0,
                        drop_last=False,
                        verbose=1
                    )
                    
                else:
                    # 后续轮次：尝试从上一个检查点继续
                    print("    继续训练...")
                    
                    # 由于TabNet的warm_start可能不可靠，我们有两个选择：
                    # 选择1：重新加载模型（更可靠但可能丢失优化器状态）
                    if False:  # 设置为True以使用重新加载方法
                        last_checkpoint = self.checkpoint_manager.checkpoint_info[-1]
                        print(f"    从检查点重新加载: {last_checkpoint['model_path']}")
                        
                        # 重新构建模型
                        self.build_model()
                        
                        # 加载权重
                        self.model.load_model(last_checkpoint['model_path'])
                    
                    # 选择2：尝试warm_start（可能不完全有效）
                    try:
                        self.model.fit(
                            X_train=self.X_train,
                            y_train=self.y_train,
                            eval_set=[(self.X_val, self.y_val)],
                            eval_name=['val'],
                            eval_metric=['accuracy', 'logloss', MacroF1],
                            max_epochs=epochs_this_round,
                            patience=0,
                            batch_size=self.config['batch_size'],
                            virtual_batch_size=virtual_batch_size,
                            num_workers=0,
                            drop_last=False,
                            warm_start=True,  # 尝试warm start
                            verbose=1
                        )
                    except Exception as e:
                        print(f"    ⚠️ Warm start失败，使用标准训练: {e}")
                        self.model.fit(
                            X_train=self.X_train,
                            y_train=self.y_train,
                            eval_set=[(self.X_val, self.y_val)],
                            eval_name=['val'],
                            eval_metric=['accuracy', 'logloss', MacroF1],
                            max_epochs=epochs_this_round,
                            patience=0,
                            batch_size=self.config['batch_size'],
                            virtual_batch_size=virtual_batch_size,
                            num_workers=0,
                            drop_last=False,
                            verbose=1
                        )
                
                # 更新当前epoch
                current_epoch = end_epoch
                
                # 评估当前性能
                print(f"\n📊 Epoch {current_epoch} - 评估三数据集性能...")
                
                # 使用monitor进行完整评估
                val_f1 = self.monitor.on_epoch_end(
                    epoch=current_epoch - 1,
                    model=self.model,
                    X_train=self.X_train,
                    y_train=self.y_train,
                    X_val=self.X_val,
                    y_val=self.y_val,
                    X_test=self.X_test,
                    y_test=self.y_test
                )
                
                # 获取详细的性能指标用于检查点保存
                training_history_snapshot = {
                    'epoch': current_epoch,
                    'train_f1_macro': self.monitor.history['train_f1_macro'][-1] if self.monitor.history['train_f1_macro'] else None,
                    'val_f1_macro': self.monitor.history['val_f1_macro'][-1] if self.monitor.history['val_f1_macro'] else None,
                    'test_f1_macro': self.monitor.history['test_f1_macro'][-1] if self.monitor.history['test_f1_macro'] else None,
                    'train_loss': self.monitor.history['train_loss'][-1] if self.monitor.history['train_loss'] else None,
                    'val_loss': self.monitor.history['val_loss'][-1] if self.monitor.history['val_loss'] else None,
                    'test_loss': self.monitor.history['test_loss'][-1] if self.monitor.history['test_loss'] else None,
                    'train_accuracy': self.monitor.history['train_accuracy'][-1] if self.monitor.history['train_accuracy'] else None,
                    'val_accuracy': self.monitor.history['val_accuracy'][-1] if self.monitor.history['val_accuracy'] else None,
                    'test_accuracy': self.monitor.history['test_accuracy'][-1] if self.monitor.history['test_accuracy'] else None,
                    'overfitting_gap': self.monitor.history['overfitting_gap'][-1] if self.monitor.history['overfitting_gap'] else None,
                    'generalization_gap': self.monitor.history['generalization_gap'][-1] if self.monitor.history['generalization_gap'] else None
                }
                
                # 保存检查点
                checkpoint_path = self.checkpoint_manager.save_checkpoint(
                    model=self.model,
                    epoch=current_epoch,
                    config=self.config,
                    training_history=training_history_snapshot,
                    dataset_info=dataset_info
                )
                
                print(f"💾 检查点已保存: Epoch {current_epoch}")
                
                # 手动实现early stopping
                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    patience_counter = 0
                    best_epoch = current_epoch
                    print(f"✅ 新的最佳验证F1: {val_f1:.4f}")
                    
                    # 可选：保存最佳模型的特殊标记
                    best_model_path = os.path.join(
                        self.checkpoint_manager.checkpoint_dir, 
                        "best_model.zip"
                    )
                    self.model.save_model(best_model_path)
                    print(f"🏆 最佳模型已保存: {best_model_path}")
                    
                else:
                    patience_counter += 1
                    print(f"⏳ 验证F1未改善: {patience_counter}/{self.config['patience']}")
                    
                    if patience_counter >= self.config['patience']:
                        print(f"\n🛑 Early stopping triggered at epoch {current_epoch}")
                        print(f"   最佳验证F1: {best_val_f1:.4f} (Epoch {best_epoch})")
                        break
                
                # 检查是否已达到最大epochs
                if current_epoch >= total_epochs:
                    print(f"\n📍 达到最大训练轮数: {total_epochs}")
                    break
                
                # 打印进度信息
                elapsed_time = time.time() - start_time
                epochs_remaining = total_epochs - current_epoch
                estimated_time_remaining = (elapsed_time / current_epoch) * epochs_remaining if current_epoch > 0 else 0
                
                print(f"\n⏱️ 进度: {current_epoch}/{total_epochs} epochs")
                print(f"   已用时间: {elapsed_time/60:.1f} 分钟")
                print(f"   预计剩余: {estimated_time_remaining/60:.1f} 分钟")
            
            # 训练完成后的最终评估
            print(f"\n🎯 训练完成，进行最终评估...")
            self.evaluate_all_datasets_final()
            final_val_f1 = self.evaluate_final_performance()
            
            # 保存最终检查点（如果最后一个epoch不是检查点）
            if current_epoch % checkpoint_frequency != 0:
                print(f"\n💾 保存最终检查点...")
                
                final_history_snapshot = {
                    'epoch': current_epoch,
                    'train_f1_macro': self.monitor.history['train_f1_macro'][-1],
                    'val_f1_macro': self.monitor.history['val_f1_macro'][-1],
                    'test_f1_macro': self.monitor.history['test_f1_macro'][-1],
                    'train_loss': self.monitor.history['train_loss'][-1],
                    'val_loss': self.monitor.history['val_loss'][-1],
                    'test_loss': self.monitor.history['test_loss'][-1],
                    'final_evaluation': True
                }
                
                self.checkpoint_manager.save_checkpoint(
                    model=self.model,
                    epoch=current_epoch,
                    config=self.config,
                    training_history=final_history_snapshot,
                    dataset_info=dataset_info
                )
            
            # 记录训练完成
            training_time = time.time() - start_time
            self.training_completed = True
            
            # 打印训练总结
            print(f"\n{'='*60}")
            print(f"✅ {self.config_name} 训练完成!")
            print(f"{'='*60}")
            print(f"📊 训练统计:")
            print(f"   - 实际训练epochs: {current_epoch}")
            print(f"   - 最佳epoch: {best_epoch if 'best_epoch' in locals() else current_epoch}")
            print(f"   - 最佳验证F1: {best_val_f1:.4f}")
            print(f"   - 最终验证F1: {final_val_f1:.4f}")
            print(f"   - 保存的检查点数: {len(self.checkpoint_manager.checkpoint_info)}")
            print(f"   - 检查点目录: {self.checkpoint_manager.checkpoint_dir}")
            print(f"   - 训练时间: {training_time:.2f}秒 ({training_time/60:.1f}分钟)")
            print(f"{'='*60}")
            
            return current_epoch, training_time
            
        except Exception as e:
            print(f"\n❌ {self.config_name} 训练失败: {e}")
            import traceback
            traceback.print_exc()
            
            # 即使失败也尝试保存当前状态
            if current_epoch > 0:
                try:
                    print(f"\n💾 尝试保存失败前的检查点...")
                    self.checkpoint_manager.save_checkpoint(
                        model=self.model,
                        epoch=current_epoch,
                        config=self.config,
                        training_history={'error': str(e), 'epoch': current_epoch},
                        dataset_info=dataset_info
                    )
                except:
                    pass
            
            return current_epoch, time.time() - start_time
            


    
    def train_standard(self):
        """标准训练 - 只在训练完成后显示三数据集F1"""
        
        if self.model is None:
            self.build_model()
        
        print(f"\n🚀 开始 {self.config_name} 标准训练...")
        print(f"    最大轮数: {self.config['max_epochs']}")
        print(f"    Early stopping patience: {self.config['patience']}")
        print(f"    批次大小: {self.config['batch_size']}")
        print(f"    ⏰ 只在训练完成后显示三数据集F1")
        
        # 计算批次参数
        virtual_batch_size = max(32, int(self.config['batch_size'] * self.config['virtual_batch_size_ratio']))
        
        start_time = time.time()
        
        try:
            print(f"    🔧 批次大小: {self.config['batch_size']}, 虚拟批次: {virtual_batch_size}")
            
            # 创建自定义F1 metric for TabNet
            from pytorch_tabnet.metrics import Metric
            from sklearn.metrics import f1_score as sklearn_f1
            
            class MacroF1(Metric):
                def __init__(self):
                    self._name = "macro_f1"
                    self._maximize = True
                
                def __call__(self, y_true, y_score):
                    # y_score是概率，需要转换为预测标签
                    y_pred = np.argmax(y_score, axis=1)
                    return sklearn_f1(y_true, y_pred, average='macro')
            
            # 训练模型 - 添加自定义F1 metric
            self.model.fit(
                X_train=self.X_train,
                y_train=self.y_train,
                eval_set=[(self.X_val, self.y_val)],
                eval_name=['val'],
                eval_metric=['accuracy', 'logloss', MacroF1],  # 添加F1 metric
                max_epochs=self.config['max_epochs'],
                patience=self.config['patience'],
                batch_size=self.config['batch_size'],
                virtual_batch_size=virtual_batch_size,
                num_workers=0,
                drop_last=False
            )
            
            # 训练完成后进行完整的三数据集评估
            print(f"\n🎯 {self.config_name} 训练完成，进行完整三数据集评估...")
            self.evaluate_all_datasets_final()
            
            # 最终评估
            final_val_f1 = self.evaluate_final_performance()
            
            training_time = time.time() - start_time
            self.training_completed = True
            
            print(f"✅ {self.config_name} 训练完成!")
            print(f"    训练时间: {training_time:.2f}秒 ({training_time/60:.1f}分钟)")
            print(f"    最终验证F1: {final_val_f1:.4f}")
            
            return final_val_f1, training_time
            
        except Exception as e:
            print(f"❌ {self.config_name} 训练失败: {e}")
            import traceback
            traceback.print_exc()
            return 0.0, 0.0
    
    def evaluate_single_epoch(self, epoch):
        """单个epoch的三数据集评估"""
        
        try:
            print(f"\n📊 Epoch {epoch + 1} - 三数据集评估:")
            print("-" * 45)
            
            # 预测三个数据集
            train_preds = self.model.predict(self.X_train)
            train_proba = self.model.predict_proba(self.X_train)
            
            val_preds = self.model.predict(self.X_val)
            val_proba = self.model.predict_proba(self.X_val)
            
            test_preds = self.model.predict(self.X_test)
            test_proba = self.model.predict_proba(self.X_test)
            
            # 计算F1 scores
            train_f1 = f1_score(self.y_train, train_preds, average='macro')
            val_f1 = f1_score(self.y_val, val_preds, average='macro')
            test_f1 = f1_score(self.y_test, test_preds, average='macro')
            
            # 计算准确率
            train_acc = accuracy_score(self.y_train, train_preds)
            val_acc = accuracy_score(self.y_val, val_preds)
            test_acc = accuracy_score(self.y_test, test_preds)
            
            # 计算损失
            train_loss = self.monitor.calculate_cross_entropy_loss(self.y_train, train_proba)
            val_loss = self.monitor.calculate_cross_entropy_loss(self.y_val, val_proba)
            test_loss = self.monitor.calculate_cross_entropy_loss(self.y_test, test_proba)
            
            # 计算gap指标
            overfitting_gap = train_f1 - val_f1
            generalization_gap = val_f1 - test_f1
            
            # 漂亮的表格式输出
            print(f"📈 F1      | Train: {train_f1:.4f} | Val: {val_f1:.4f} | Test: {test_f1:.4f}")
            print(f"🎯 Acc     | Train: {train_acc:.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f}")
            print(f"📉 Loss    | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Test: {test_loss:.4f}")
            print(f"🔍 Gaps    | Overfit: {overfitting_gap:+.4f} | General: {generalization_gap:+.4f}")
            
            # 状态指示
            if overfitting_gap < 0.05:
                status = "✅"
            elif overfitting_gap < 0.10:
                status = "⚠️"
            else:
                status = "❌"
            
            trend = "📈" if len(self.monitor.history['val_f1_macro']) == 0 or val_f1 > max(self.monitor.history['val_f1_macro']) else "📉"
            print(f"📊 Status  | {status} | Val F1 trend: {trend}")
            
            # 记录历史
            self.monitor.history['epoch'].append(epoch + 1)
            self.monitor.history['train_f1_macro'].append(train_f1)
            self.monitor.history['val_f1_macro'].append(val_f1)
            self.monitor.history['test_f1_macro'].append(test_f1)
            self.monitor.history['train_loss'].append(train_loss)
            self.monitor.history['val_loss'].append(val_loss)
            self.monitor.history['test_loss'].append(test_loss)
            self.monitor.history['train_accuracy'].append(train_acc)
            self.monitor.history['val_accuracy'].append(val_acc)
            self.monitor.history['test_accuracy'].append(test_acc)
            self.monitor.history['overfitting_gap'].append(overfitting_gap)
            self.monitor.history['generalization_gap'].append(generalization_gap)
            
            return val_f1
            
        except Exception as e:
            print(f"❌ Epoch {epoch + 1} 评估失败: {e}")
            return 0.0

    def evaluate_all_datasets_final(self):
        """训练后立即评估所有三个数据集"""
        
        print(f"\n📊 {self.config_name} 三数据集详细评估:")
        print("="*50)
        
        try:
            # 预测三个数据集
            train_preds = self.model.predict(self.X_train)
            train_proba = self.model.predict_proba(self.X_train)
            
            val_preds = self.model.predict(self.X_val)
            val_proba = self.model.predict_proba(self.X_val)
            
            test_preds = self.model.predict(self.X_test)
            test_proba = self.model.predict_proba(self.X_test)
            
            # 计算F1 scores
            train_f1 = f1_score(self.y_train, train_preds, average='macro')
            val_f1 = f1_score(self.y_val, val_preds, average='macro')
            test_f1 = f1_score(self.y_test, test_preds, average='macro')
            
            # 计算准确率
            train_acc = accuracy_score(self.y_train, train_preds)
            val_acc = accuracy_score(self.y_val, val_preds)
            test_acc = accuracy_score(self.y_test, test_preds)
            
            # 计算损失
            train_loss = self.monitor.calculate_cross_entropy_loss(self.y_train, train_proba)
            val_loss = self.monitor.calculate_cross_entropy_loss(self.y_val, val_proba)
            test_loss = self.monitor.calculate_cross_entropy_loss(self.y_test, test_proba)
            
            # 计算gap指标
            overfitting_gap = train_f1 - val_f1
            generalization_gap = val_f1 - test_f1
            
            # 打印详细结果
            print(f"📈 Macro F1 Scores:")
            print(f"   🔹 训练集: {train_f1:.4f}")
            print(f"   🔹 验证集: {val_f1:.4f}")
            print(f"   🔹 测试集: {test_f1:.4f} (观察用)")
            
            print(f"\n🎯 准确率:")
            print(f"   🔹 训练集: {train_acc:.4f}")
            print(f"   🔹 验证集: {val_acc:.4f}")
            print(f"   🔹 测试集: {test_acc:.4f}")
            
            print(f"\n📉 交叉熵损失:")
            print(f"   🔹 训练集: {train_loss:.4f}")
            print(f"   🔹 验证集: {val_loss:.4f}")
            print(f"   🔹 测试集: {test_loss:.4f}")
            
            print(f"\n🔍 性能分析:")
            print(f"   🔹 过拟合Gap (Train-Val F1): {overfitting_gap:+.4f}")
            print(f"   🔹 泛化Gap (Val-Test F1): {generalization_gap:+.4f}")
            
            # 简单的状态评估
            if overfitting_gap < 0.05:
                overfit_status = "✅ 良好"
            elif overfitting_gap < 0.10:
                overfit_status = "⚠️ 轻微过拟合"
            else:
                overfit_status = "❌ 明显过拟合"
            
            if abs(generalization_gap) < 0.03:
                general_status = "✅ 泛化良好"
            elif abs(generalization_gap) < 0.06:
                general_status = "⚠️ 泛化一般"
            else:
                general_status = "❌ 泛化较差"
            
            print(f"   🔹 过拟合状态: {overfit_status}")
            print(f"   🔹 泛化状态: {general_status}")
            
            # 记录到monitor历史中
            self.monitor.history['train_f1_macro'].append(train_f1)
            self.monitor.history['val_f1_macro'].append(val_f1)
            self.monitor.history['test_f1_macro'].append(test_f1)
            self.monitor.history['train_loss'].append(train_loss)
            self.monitor.history['val_loss'].append(val_loss)
            self.monitor.history['test_loss'].append(test_loss)
            self.monitor.history['train_accuracy'].append(train_acc)
            self.monitor.history['val_accuracy'].append(val_acc)
            self.monitor.history['test_accuracy'].append(test_acc)
            self.monitor.history['overfitting_gap'].append(overfitting_gap)
            self.monitor.history['generalization_gap'].append(generalization_gap)
            
        except Exception as e:
            print(f"❌ 三数据集评估失败: {e}")
    
    def evaluate_final_performance(self):
        """最终性能评估"""
        
        try:
            # 预测三个数据集
            train_preds = self.model.predict(self.X_train)
            train_proba = self.model.predict_proba(self.X_train)
            
            val_preds = self.model.predict(self.X_val)
            val_proba = self.model.predict_proba(self.X_val)
            
            test_preds = self.model.predict(self.X_test)
            test_proba = self.model.predict_proba(self.X_test)
            
            # 计算完整指标
            results = {
                'train_f1_macro': f1_score(self.y_train, train_preds, average='macro'),
                'val_f1_macro': f1_score(self.y_val, val_preds, average='macro'),
                'test_f1_macro': f1_score(self.y_test, test_preds, average='macro'),
                'train_accuracy': accuracy_score(self.y_train, train_preds),
                'val_accuracy': accuracy_score(self.y_val, val_preds),
                'test_accuracy': accuracy_score(self.y_test, test_preds),
                'train_balanced_acc': balanced_accuracy_score(self.y_train, train_preds),  
                'val_balanced_acc': balanced_accuracy_score(self.y_val, val_preds),
                'test_balanced_acc': balanced_accuracy_score(self.y_test, test_preds),
                'train_kappa': cohen_kappa_score(self.y_train, train_preds),
                'val_kappa': cohen_kappa_score(self.y_val, val_preds),
                'test_kappa': cohen_kappa_score(self.y_test, test_preds)
            }
            
            # 计算损失
            results['train_loss'] = self.monitor.calculate_cross_entropy_loss(self.y_train, train_proba)
            results['val_loss'] = self.monitor.calculate_cross_entropy_loss(self.y_val, val_proba)
            results['test_loss'] = self.monitor.calculate_cross_entropy_loss(self.y_test, test_proba)
            
            # 计算gap指标
            results['overfitting_gap'] = results['train_f1_macro'] - results['val_f1_macro']
            results['generalization_gap'] = results['val_f1_macro'] - results['test_f1_macro']
            
            # 特征重要性
            try:
                results['feature_importance'] = self.model.feature_importances_.copy()
            except:
                results['feature_importance'] = None
            
            # 保存最终结果
            self.final_results = results
            
            # 打印详细结果
            print(f"\n📊 {self.config_name} 最终详细结果:")
            print(f"   F1 Macro    - Train: {results['train_f1_macro']:.4f}, Val: {results['val_f1_macro']:.4f}, Test: {results['test_f1_macro']:.4f}")
            print(f"   Accuracy    - Train: {results['train_accuracy']:.4f}, Val: {results['val_accuracy']:.4f}, Test: {results['test_accuracy']:.4f}")
            print(f"   Balanced Acc- Train: {results['train_balanced_acc']:.4f}, Val: {results['val_balanced_acc']:.4f}, Test: {results['test_balanced_acc']:.4f}")
            print(f"   Kappa       - Train: {results['train_kappa']:.4f}, Val: {results['val_kappa']:.4f}, Test: {results['test_kappa']:.4f}")
            print(f"   Loss        - Train: {results['train_loss']:.4f}, Val: {results['val_loss']:.4f}, Test: {results['test_loss']:.4f}")
            print(f"   过拟合Gap   : {results['overfitting_gap']:+.4f}")
            print(f"   泛化Gap     : {results['generalization_gap']:+.4f}")
            
            return results['val_f1_macro']
            
        except Exception as e:
            print(f"❌ 最终评估失败: {e}")
            return 0.0
    
    def save_model(self, save_dir):
        """保存模型"""
        
        if self.model is None or not self.training_completed:
            print(f"⚠️ {self.config_name} 模型尚未训练完成，无法保存")
            return None
        
        try:
            model_path = os.path.join(save_dir, f'{self.config_name}_final_model')
            os.makedirs(os.path.dirname(model_path), exist_ok=True)
            
            saved_path = self.model.save_model(model_path)
            print(f"💾 {self.config_name} 模型已保存: {saved_path}")
            return saved_path
            
        except Exception as e:
            print(f"❌ {self.config_name} 模型保存失败: {e}")
            return None
    
    def get_candidate_info(self):
        """获取候选模型信息"""
        
        return {
            'config_name': self.config_name,
            'config': self.config,
            'final_results': getattr(self, 'final_results', {}),
            'training_completed': self.training_completed,
            'model': self.model
        }

# 🏆 最终模型选择器
class FinalModelSelector:
    """最终模型选择系统"""
    
    def __init__(self):
        self.candidates = []
        
    def add_candidate(self, trainer):
        """添加候选模型"""
        
        candidate_info = trainer.get_candidate_info()
        
        if trainer.training_completed and hasattr(trainer, 'final_results'):
            results = trainer.final_results
            
            candidate = {
                'config_name': candidate_info['config_name'],
                'config': candidate_info['config'],
                'model': candidate_info['model'],
                'trainer': trainer,
                
                # 关键性能指标
                'val_f1_macro': results['val_f1_macro'],
                'test_f1_macro': results['test_f1_macro'],
                'val_accuracy': results['val_accuracy'],
                'test_accuracy': results['test_accuracy'],
                'val_balanced_acc': results['val_balanced_acc'],
                'val_loss': results['val_loss'],
                'test_loss': results['test_loss'],
                
                # Gap分析
                'overfitting_gap': results['overfitting_gap'],
                'generalization_gap': results['generalization_gap'],
                
                # 完整结果
                'final_results': results,
                
                # 可解释性
                'feature_importance': results.get('feature_importance', None)
            }
            
            self.candidates.append(candidate)
            print(f"✅ 添加候选模型: {candidate['config_name']}")
            print(f"   验证F1: {candidate['val_f1_macro']:.4f}")
            print(f"   测试F1: {candidate['test_f1_macro']:.4f}")
            
        else:
            print(f"⚠️ {candidate_info['config_name']} 训练未完成，跳过添加")
    
    def select_final_model(self, selection_criteria='val_f1'):
        """选择最终模型"""
        
        if not self.candidates:
            print("❌ 没有可用的候选模型")
            return None
        
        print(f"\n🏆 开始最终模型选择 (标准: {selection_criteria})...")
        
        if selection_criteria == 'val_f1':
            # 基于验证F1选择
            best_candidate = max(self.candidates, key=lambda x: x['val_f1_macro'])
            print(f"   选择标准: 最高验证F1")
            
        elif selection_criteria == 'robust':
            # 基于综合评分选择（验证F1 - 过拟合惩罚）
            scores = []
            for candidate in self.candidates:
                # 综合评分：验证F1 - 过拟合惩罚 - 泛化惩罚
                overfitting_penalty = max(0, candidate['overfitting_gap'] - 0.05) * 0.2
                generalization_penalty = max(0, abs(candidate['generalization_gap']) - 0.03) * 0.1
                score = candidate['val_f1_macro'] - overfitting_penalty - generalization_penalty
                scores.append(score)
                
                print(f"   {candidate['config_name']}: 验证F1={candidate['val_f1_macro']:.4f}, "
                      f"过拟合惩罚={overfitting_penalty:.4f}, 泛化惩罚={generalization_penalty:.4f}, "
                      f"综合得分={score:.4f}")
            
            best_idx = np.argmax(scores)
            best_candidate = self.candidates[best_idx]
            print(f"   选择标准: 综合得分 (考虑过拟合和泛化)")
        
        print(f"\n🥇 最终选择: {best_candidate['config_name']}")
        print(f"   验证F1: {best_candidate['val_f1_macro']:.4f}")
        print(f"   测试F1: {best_candidate['test_f1_macro']:.4f}")
        print(f"   验证-测试Gap: {best_candidate['generalization_gap']:+.4f}")
        print(f"   过拟合Gap: {best_candidate['overfitting_gap']:+.4f}")
        
        return best_candidate
    
    def generate_comparison_report(self, save_path=None):
        """生成模型对比报告"""
        
        if not self.candidates:
            print("❌ 没有候选模型，无法生成报告")
            return
        
        print(f"\n📋 生成模型对比报告...")
        
        report_lines = []
        report_lines.append("TabNet 5层级配置简化实验报告")
        report_lines.append("=" * 60)
        report_lines.append(f"生成时间: {time.strftime('%Y-%m-%d %H:%M:%S')}")
        report_lines.append(f"设备: {device}")
        report_lines.append(f"候选模型数量: {len(self.candidates)}")
        report_lines.append("")
        
        # 数据集信息
        report_lines.append("数据集信息:")
        report_lines.append("-" * 30)
        report_lines.append(f"训练集样本数: {len(self.candidates[0]['trainer'].X_train)}")
        report_lines.append(f"验证集样本数: {len(self.candidates[0]['trainer'].X_val)}")
        report_lines.append(f"测试集样本数: {len(self.candidates[0]['trainer'].X_test)}")
        report_lines.append(f"特征维度: {self.candidates[0]['trainer'].X_train.shape[1]}")
        report_lines.append(f"类别数量: {len(np.unique(self.candidates[0]['trainer'].y_train))}")
        report_lines.append("")
        
        # 模型对比表
        report_lines.append("模型性能对比:")
        report_lines.append("-" * 30)
        header = f"{'配置':<12} {'验证F1':<8} {'测试F1':<8} {'验证Acc':<8} {'平衡Acc':<8} {'过拟合Gap':<10} {'泛化Gap':<8} {'验证Loss':<8}"
        report_lines.append(header)
        report_lines.append("-" * len(header))
        
        # 按验证F1排序
        sorted_candidates = sorted(self.candidates, key=lambda x: x['val_f1_macro'], reverse=True)
        
        for i, candidate in enumerate(sorted_candidates):
            config_short = candidate['config_name'].replace('_config', '')            
            line = (f"{config_short:<12} "
                    f"{candidate['val_f1_macro']:<8.4f} "
                    f"{candidate['test_f1_macro']:<8.4f} "
                    f"{candidate['val_accuracy']:<8.4f} "
                    f"{candidate['val_balanced_acc']:<8.4f} "
                    f"{candidate['overfitting_gap']:+<10.4f} "
                    f"{candidate['generalization_gap']:+<8.4f} "
                    f"{candidate['val_loss']:<8.4f}")

            if i == 0:  # 标记最佳
                line += " 🏆"
            
            report_lines.append(line)
        
        report_lines.append("")
        
        # 详细配置信息
        report_lines.append("详细配置信息:")
        report_lines.append("-" * 30)
        for config_name, config in TABNET_COMPLETE_CONFIGS.items():
            report_lines.append(f"\n{config_name}:")
            for key, value in config.items():
                if key != 'estimated_params':
                    report_lines.append(f"  {key}: {value}")
                else:
                    report_lines.append(f"  参数量: {value}")
        
        # 保存报告
        report_text = "\n".join(report_lines)
        
        if save_path is None:
            save_path = os.path.join(export_path, 'final_comparison_report.txt')
        
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write(report_text)
        
        print(f"✅ 对比报告已保存: {save_path}")
        
        # 同时打印到控制台
        print("\n" + "="*60)
        print("📊 模型性能对比总结")
        print("="*60)
        print(header)
        print("-" * len(header))
        for i, candidate in enumerate(sorted_candidates):
            config_short = candidate['config_name'].replace('_config', '')
            print(f"{config_short:<12} "
                  f"{candidate['val_f1_macro']:<8.4f} "
                  f"{candidate['test_f1_macro']:<8.4f} "
                  f"{candidate['val_accuracy']:<8.4f} "
                  f"{candidate['overfitting_gap']:+<10.4f} "
                  f"{candidate['generalization_gap']:+<8.4f} "
                  f"{candidate['val_loss']:<8.4f}" +
                  (" 🏆" if i == 0 else ""))
        
        return save_path

# 🏆 简化的实验管理器
class SimplifiedExperimentManager:
    """简化的TabNet实验管理器"""
    
    def __init__(self, X_train, y_train, X_val, y_val, X_test, y_test):
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.X_test = X_test
        self.y_test = y_test
        self.trained_models = {}
        
    def run_complete_experiment(self, configs_to_run, checkpoint_frequency=5):
        """运行完整实验流程 - 使用检查点保存"""
        
        print(f"\n🚀 开始简化TabNet实验")
        print(f"📋 配置列表: {configs_to_run}")
        print(f"💾 检查点频率: 每{checkpoint_frequency}个epoch保存一次")
        
        model_selector = FinalModelSelector()
        checkpoint_analyzers = {}
        
        for config_name in configs_to_run:
            print(f"\n🔥 {config_name} 训练...")
            
            try:
                # 创建训练器
                trainer = SimplifiedTabNetTrainer(
                    config_name=config_name,
                    X_train=self.X_train,
                    y_train=self.y_train,
                    X_val=self.X_val,
                    y_val=self.y_val,
                    X_test=self.X_test,
                    y_test=self.y_test
                )
                
                # 使用检查点训练
                cumulative_epochs, training_time = trainer.train_with_checkpoints(
                    checkpoint_frequency=checkpoint_frequency
                )
                
                # 记录检查点目录
                checkpoint_dir = os.path.join(export_path, 'checkpoints', config_name)
                checkpoint_analyzers[config_name] = checkpoint_dir
                
                self.trained_models[config_name] = {
                    'trainer': trainer,
                    'cumulative_epochs': cumulative_epochs,
                    'training_time': training_time,
                    'checkpoint_dir': checkpoint_dir
                }
                
                print(f"✅ {config_name} 训练完成: {cumulative_epochs} epochs, 用时={training_time/60:.1f}分钟")
                
            except Exception as e:
                print(f"❌ {config_name} 训练失败: {e}")
                continue
        
        # 后期分析所有检查点
        print(f"\n📊 开始分析所有配置的检查点...")
        
        for config_name, checkpoint_dir in checkpoint_analyzers.items():
            print(f"\n🔍 分析 {config_name} 的检查点...")
            
            analyzer = CheckpointAnalyzer(
                checkpoint_dir=checkpoint_dir,
                X_train=self.X_train,
                y_train=self.y_train,
                X_val=self.X_val,
                y_val=self.y_val,
                X_test=self.X_test,
                y_test=self.y_test
            )
            
            # 分析所有检查点
            results_df = analyzer.analyze_all_checkpoints()
            
            # 基于分析结果选择最佳模型
            best_idx = results_df['val_f1_macro'].idxmax()
            best_checkpoint = results_df.loc[best_idx]
            
            print(f"\n🏆 {config_name} 最佳检查点:")
            print(f"  Epoch: {best_checkpoint['epoch']}")
            print(f"  验证F1: {best_checkpoint['val_f1_macro']:.4f}")
            print(f"  测试F1: {best_checkpoint['test_f1_macro']:.4f}")
            
        # 在run_complete_experiment方法的最后，添加返回值
        if checkpoint_analyzers:
            # 创建一个临时的model_selector来返回
            # 基于最后的检查点分析结果
            final_model = None
            all_candidates = []
            
            for config_name in checkpoint_analyzers:
                if config_name in self.trained_models:
                    trainer = self.trained_models[config_name]['trainer']
                    # 添加到候选列表
                    model_selector.add_candidate(trainer)
            
            # 选择最终模型
            if model_selector.candidates:
                final_model = model_selector.select_final_model(selection_criteria='val_f1')
                all_candidates = model_selector.candidates
            
            return final_model, all_candidates
        else:
            return None, []

    
    def generate_all_reports_and_visualizations(self, final_model, all_candidates):
        """生成所有报告和可视化"""
        
        print(f"\n📊 生成完整报告和可视化...")
        
        # 创建临时选择器来生成报告
        temp_selector = FinalModelSelector()
        temp_selector.candidates = all_candidates
        
        # 生成对比报告
        report_path = temp_selector.generate_comparison_report()
        
        # 生成可视化
        viz_path = plot_complete_experiment_results(
            all_candidates, final_model, 
            os.path.join(export_path, 'visualizations')
        )
        
        # 生成实验总结
        summary_path = self.generate_experiment_summary(final_model, all_candidates)
        
        return {
            'report_path': report_path,
            'visualization_path': viz_path,
            'summary_path': summary_path
        }
    
    def generate_experiment_summary(self, final_model, all_candidates):
        """生成实验总结"""
        
        summary_lines = []
        summary_lines.append("TabNet简化实验总结")
        summary_lines.append("=" * 50)
        summary_lines.append(f"生成时间: {time.strftime('%Y-%m-%d %H:%M:%S')}")
        summary_lines.append("")
        
        # 实验规模
        summary_lines.append("实验规模:")
        summary_lines.append(f"  测试配置数量: {len(self.trained_models)}")
        summary_lines.append(f"  成功训练模型: {len(all_candidates)}")
        summary_lines.append(f"  使用预定义超参数")
        summary_lines.append("")
        
        # 最佳模型信息
        summary_lines.append("最佳模型:")
        summary_lines.append(f"  配置: {final_model['config_name']}")
        summary_lines.append(f"  验证F1: {final_model['val_f1_macro']:.4f}")
        summary_lines.append(f"  测试F1: {final_model['test_f1_macro']:.4f}")
        summary_lines.append(f"  参数量: {TABNET_COMPLETE_CONFIGS[final_model['config_name']]['estimated_params']}")
        summary_lines.append("")
        
        # 最佳超参数
        best_config = TABNET_COMPLETE_CONFIGS[final_model['config_name']]
        summary_lines.append("最佳超参数:")
        for key, value in best_config.items():
            if key not in ['estimated_params']:
                if isinstance(value, float):
                    summary_lines.append(f"  {key}: {value:.2e}")
                else:
                    summary_lines.append(f"  {key}: {value}")
        summary_lines.append("")
        
        # 性能分析
        val_f1s = [c['val_f1_macro'] for c in all_candidates]
        test_f1s = [c['test_f1_macro'] for c in all_candidates]
        
        summary_lines.append("性能统计:")
        summary_lines.append(f"  验证F1范围: {min(val_f1s):.4f} - {max(val_f1s):.4f}")
        summary_lines.append(f"  测试F1范围: {min(test_f1s):.4f} - {max(test_f1s):.4f}")
        summary_lines.append(f"  平均验证F1: {np.mean(val_f1s):.4f}")
        summary_lines.append(f"  平均测试F1: {np.mean(test_f1s):.4f}")
        
        summary_text = "\n".join(summary_lines)
        
        save_path = os.path.join(export_path, 'experiment_summary.txt')
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write(summary_text)
        
        print(f"✅ 实验总结已保存: {save_path}")
        return save_path

# 📊 完整可视化系统
def plot_complete_experiment_results(candidates, final_model, save_dir):
    """绘制完整实验结果的可视化"""
    
    print(f"\n📊 生成完整实验可视化...")
    
    # 创建超大图表
    fig = plt.figure(figsize=(24, 16))
    
    # 1. 性能对比雷达图
    ax1 = plt.subplot(3, 4, 1, projection='polar')
    plot_performance_radar(candidates, ax1)
    
    # 2. 验证F1对比柱状图
    ax2 = plt.subplot(3, 4, 2)
    plot_validation_f1_comparison(candidates, final_model, ax2)
    
    # 3. 测试F1对比柱状图
    ax3 = plt.subplot(3, 4, 3)
    plot_test_f1_comparison(candidates, final_model, ax3)
    
    # 4. 过拟合分析
    ax4 = plt.subplot(3, 4, 4)
    plot_overfitting_analysis(candidates, ax4)
    
    # 5. 损失对比
    ax5 = plt.subplot(3, 4, 5)
    plot_loss_comparison(candidates, ax5)
    
    # 6. 参数量vs性能散点图
    ax6 = plt.subplot(3, 4, 6)
    plot_params_vs_performance(candidates, ax6)
    
    # 7. 配置参数热力图
    ax7 = plt.subplot(3, 4, 7)
    plot_config_heatmap(candidates, ax7)
    
    # 8. 泛化性能分析
    ax8 = plt.subplot(3, 4, 8)
    plot_generalization_analysis(candidates, ax8)
    
    # 9-12. 特征重要性（前4个模型）
    top_4_candidates = sorted(candidates, key=lambda x: x['val_f1_macro'], reverse=True)[:4]
    for i, candidate in enumerate(top_4_candidates):
        ax = plt.subplot(3, 4, 9 + i)
        plot_feature_importance_single(candidate, ax, top_k=15)
    
    plt.tight_layout()
    
    # 保存图表
    save_path = os.path.join(save_dir, 'complete_experiment_visualization.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ 完整可视化已保存: {save_path}")
    
    return save_path

def plot_performance_radar(candidates, ax):
    """绘制性能雷达图"""
    
    # 选择所有候选模型
    top_candidates = sorted(candidates, key=lambda x: x['val_f1_macro'], reverse=True)
    
    # 定义指标
    metrics = ['Val F1', 'Test F1', 'Val Acc', 'Robustness', 'Efficiency']
    
    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]  # 闭合雷达图
    
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    
    for i, candidate in enumerate(top_candidates):
        # 计算各项指标（归一化到0-1）
        val_f1_norm = candidate['val_f1_macro']
        test_f1_norm = candidate['test_f1_macro'] 
        val_acc_norm = candidate['val_accuracy']
        robustness = max(0, 1 - abs(candidate['overfitting_gap']) / 0.2)  # 过拟合越小越好
        efficiency = 1.0  # 假设都是1，实际可以根据训练时间计算
        
        values = [val_f1_norm, test_f1_norm, val_acc_norm, robustness, efficiency]
        values += values[:1]  # 闭合
        
        config_name = candidate['config_name'].replace('_config', '')
        color = colors[i] if i < len(colors) else colors[i % len(colors)]
        ax.plot(angles, values, 'o-', linewidth=2, label=config_name, color=color)
        ax.fill(angles, values, alpha=0.1, color=color)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics)
    ax.set_ylim(0, 1)
    ax.set_title('Performance Radar Chart', size=12, fontweight='bold')
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    ax.grid(True)

def plot_validation_f1_comparison(candidates, final_model, ax):
    """验证F1对比柱状图"""
    
    config_names = [c['config_name'].replace('_config', '') for c in candidates]
    val_f1s = [c['val_f1_macro'] for c in candidates]
    
    # 创建颜色，突出显示最终选择的模型
    colors = ['gold' if c['config_name'] == final_model['config_name'] else 'skyblue' 
              for c in candidates]
    
    bars = ax.bar(config_names, val_f1s, color=colors, alpha=0.8, edgecolor='black')
    
    # 添加数值标签
    for bar, val in zip(bars, val_f1s):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.001,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
    
    ax.set_ylabel('Validation F1 Score')
    ax.set_title('Validation F1 Comparison', fontweight='bold')
    ax.set_ylim(0, max(val_f1s) * 1.1)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 标记最终选择
    final_idx = [i for i, c in enumerate(candidates) if c['config_name'] == final_model['config_name']][0]
    ax.annotate('SELECTED', xy=(final_idx, val_f1s[final_idx]), 
                xytext=(final_idx, val_f1s[final_idx] + 0.02),
                ha='center', fontweight='bold', color='red',
                arrowprops=dict(arrowstyle='->', color='red'))

def plot_test_f1_comparison(candidates, final_model, ax):
    """测试F1对比柱状图"""
    
    config_names = [c['config_name'].replace('_config', '') for c in candidates]
    test_f1s = [c['test_f1_macro'] for c in candidates]
    
    colors = ['gold' if c['config_name'] == final_model['config_name'] else 'lightcoral' 
              for c in candidates]
    
    bars = ax.bar(config_names, test_f1s, color=colors, alpha=0.8, edgecolor='black')
    
    for bar, val in zip(bars, test_f1s):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.001,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
    
    ax.set_ylabel('Test F1 Score')
    ax.set_title('Test F1 Comparison\n(Not Used for Selection)', fontweight='bold')
    ax.set_ylim(0, max(test_f1s) * 1.1)
    ax.grid(True, alpha=0.3, axis='y')

def plot_overfitting_analysis(candidates, ax):
    """过拟合分析"""
    
    config_names = [c['config_name'].replace('_config', '') for c in candidates]
    overfitting_gaps = [c['overfitting_gap'] for c in candidates]
    
    # 颜色编码：绿色=健康，黄色=警告，红色=过拟合
    colors = []
    for gap in overfitting_gaps:
        if gap < 0.02:
            colors.append('green')
        elif gap < 0.05:
            colors.append('orange')
        else:
            colors.append('red')
    
    bars = ax.bar(config_names, overfitting_gaps, color=colors, alpha=0.7, edgecolor='black')
    
    # 添加阈值线
    ax.axhline(y=0.02, color='orange', linestyle='--', alpha=0.7, label='Warning (2%)')
    ax.axhline(y=0.05, color='red', linestyle='--', alpha=0.7, label='Overfitting (5%)')
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    
    # 添加数值标签
    for bar, val in zip(bars, overfitting_gaps):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.002,
                f'{val:+.3f}', ha='center', va='bottom', fontweight='bold')
    
    ax.set_ylabel('Train F1 - Val F1')
    ax.set_title('Overfitting Analysis', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

def plot_loss_comparison(candidates, ax):
    """损失对比"""
    
    config_names = [c['config_name'].replace('_config', '') for c in candidates]
    val_losses = [c['val_loss'] for c in candidates]
    test_losses = [c['test_loss'] for c in candidates]
    
    x = np.arange(len(config_names))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, val_losses, width, label='Validation Loss', 
                   color='skyblue', alpha=0.8, edgecolor='black')
    bars2 = ax.bar(x + width/2, test_losses, width, label='Test Loss', 
                   color='lightcoral', alpha=0.8, edgecolor='black')
    
    ax.set_ylabel('Cross Entropy Loss')
    ax.set_title('Loss Comparison', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(config_names)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

def plot_params_vs_performance(candidates, ax):
    """参数量vs性能散点图"""
    
    # 估算参数量（简化版）
    param_estimates = {
        'micro_config': 1,
        'small_config': 3,
        'medium_config': 8,
        'large_config': 20,
        'xlarge_config': 35
    }
    
    param_counts = [param_estimates.get(c['config_name'], 10) for c in candidates]
    val_f1s = [c['val_f1_macro'] for c in candidates]
    config_names = [c['config_name'].replace('_config', '') for c in candidates]
    
    # 创建气泡图（大小表示测试F1）
    test_f1s = [c['test_f1_macro'] for c in candidates]
    bubble_sizes = [f1 * 1000 for f1 in test_f1s]  # 放大以便可见
    
    scatter = ax.scatter(param_counts, val_f1s, s=bubble_sizes, alpha=0.6, 
                        c=range(len(candidates)), cmap='viridis', edgecolor='black')
    
    # 添加标签
    for i, (x, y, name) in enumerate(zip(param_counts, val_f1s, config_names)):
        ax.annotate(name, (x, y), xytext=(5, 5), textcoords='offset points',
                   fontsize=10, fontweight='bold')
    
    ax.set_xlabel('Estimated Parameters (Millions)')
    ax.set_ylabel('Validation F1 Score')
    ax.set_title('Model Size vs Performance\n(Bubble size = Test F1)', fontweight='bold')
    ax.grid(True, alpha=0.3)

def plot_config_heatmap(candidates, ax):
    """配置参数热力图"""
    
    # 选择关键超参数进行可视化
    config_names = [c['config_name'].replace('_config', '') for c in candidates]
    
    # 提取关键参数
    params_to_plot = ['learning_rate', 'gamma', 'lambda_sparse', 'n_steps', 'n_d']
    param_matrix = []
    
    for candidate in candidates:
        config = candidate['config']
        row = []
        for param in params_to_plot:
            if param in config:
                # 标准化参数值到0-1范围以便比较
                if param == 'learning_rate':
                    row.append(np.log10(config[param]) / np.log10(1e-2))  # 相对于最大值1e-2
                elif param == 'lambda_sparse':
                    row.append(np.log10(config[param]) / np.log10(1e-2))  # 相对于最大值1e-2
                elif param == 'gamma':
                    row.append((config[param] - 1.0) / 1.0)  # 相对于范围1-2
                elif param == 'n_steps':
                    row.append((config[param] - 2) / 4.0)  # 相对于范围2-6
                elif param == 'n_d':
                    row.append(np.log2(config[param]) / np.log2(256))  # 相对于最大值256
                else:
                    row.append(config[param])
            else:
                row.append(0)
        param_matrix.append(row)
    
    param_matrix = np.array(param_matrix)
    
    # 绘制热力图
    im = ax.imshow(param_matrix.T, cmap='viridis', aspect='auto')
    
    # 设置标签
    ax.set_xticks(range(len(config_names)))
    ax.set_xticklabels(config_names)
    ax.set_yticks(range(len(params_to_plot)))
    ax.set_yticklabels(params_to_plot)
    
    # 添加数值标签
    for i in range(len(config_names)):
        for j in range(len(params_to_plot)):
            text = ax.text(i, j, f'{param_matrix[i, j]:.2f}',
                          ha="center", va="center", color="white", fontweight='bold')
    
    ax.set_title('Configuration Parameters Heatmap', fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.6)

def plot_generalization_analysis(candidates, ax):
    """泛化性能分析"""
    
    config_names = [c['config_name'].replace('_config', '') for c in candidates]
    generalization_gaps = [c['generalization_gap'] for c in candidates]
    
    # 颜色编码：绿色=良好泛化，黄色=一般，红色=泛化差
    colors = []
    for gap in generalization_gaps:
        if abs(gap) < 0.02:
            colors.append('green')
        elif abs(gap) < 0.05:
            colors.append('orange')
        else:
            colors.append('red')
    
    bars = ax.bar(config_names, generalization_gaps, color=colors, alpha=0.7, edgecolor='black')
    
    # 添加阈值线
    ax.axhline(y=0.02, color='orange', linestyle='--', alpha=0.7, label='Good (±2%)')
    ax.axhline(y=-0.02, color='orange', linestyle='--', alpha=0.7)
    ax.axhline(y=0.05, color='red', linestyle='--', alpha=0.7, label='Poor (±5%)')
    ax.axhline(y=-0.05, color='red', linestyle='--', alpha=0.7)
    ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    
    # 添加数值标签
    for bar, val in zip(bars, generalization_gaps):
        height = bar.get_height()
        va = 'bottom' if height >= 0 else 'top'
        y_offset = 0.002 if height >= 0 else -0.002
        ax.text(bar.get_x() + bar.get_width()/2., height + y_offset,
                f'{val:+.3f}', ha='center', va=va, fontweight='bold')
    
    ax.set_ylabel('Val F1 - Test F1')
    ax.set_title('Generalization Analysis', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

def plot_feature_importance_single(candidate, ax, top_k=15):
    """绘制单个模型的特征重要性"""
    
    config_name = candidate['config_name'].replace('_config', '')
    feature_importance = candidate['feature_importance']
    
    if feature_importance is not None:
        # 获取top_k重要特征
        top_indices = np.argsort(feature_importance)[-top_k:][::-1]
        top_importance = feature_importance[top_indices]
        
        # 绘制水平柱状图
        y_pos = np.arange(len(top_importance))
        bars = ax.barh(y_pos, top_importance, color='steelblue', alpha=0.7, edgecolor='black')
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels([f'Feature {idx}' for idx in top_indices])
        ax.set_xlabel('Importance')
        ax.set_title(f'{config_name}\nTop {top_k} Features', fontweight='bold')
        ax.grid(True, alpha=0.3, axis='x')
        
        # 翻转y轴使最重要的在顶部
        ax.invert_yaxis()
        
    else:
        ax.text(0.5, 0.5, f'No feature importance\navailable for\n{config_name}', 
                ha='center', va='center', transform=ax.transAxes,
                fontsize=12, bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.5))
        ax.set_title(f'{config_name}\nFeature Importance', fontweight='bold')

# 📋 主实验执行函数
def main_simplified_experiment(detailed_monitoring=False):
    """主实验执行函数 - 简化版
    
    Args:
        detailed_monitoring: 是否启用详细监控（每5个epoch显示三数据集F1）
    """
    
    print("\n" + "🚀"*20)
    print("TabNet 5层级配置简化实验开始")
    print("🚀"*20)
    
    # 验证数据
    print(f"\n📊 数据验证:")
    print(f"  训练集: {X_train_scaled.shape}")
    print(f"  验证集: {X_val_scaled.shape}")
    print(f"  测试集: {X_test_scaled.shape}")
    print(f"  类别数: {len(np.unique(y_train_int))}")
    
    # 创建实验管理器
    experiment_manager = SimplifiedExperimentManager(
        X_train=X_train_scaled,
        y_train=y_train_int,
        X_val=X_val_scaled,
        y_val=y_val_int,
        X_test=X_test_scaled,
        y_test=y_test_int
    )
    
    # 配置实验参数
    configs_to_run = ['xlarge_config', 'large_config', 'medium_config', 'small_config', 'micro_config']
    print(f"\n⚙️ 实验配置:")
    print(f"  运行配置: {[name.replace('_config', '') for name in configs_to_run]}")
    print(f"  使用预定义超参数，无需贝叶斯优化")
    if detailed_monitoring:
        print(f"  📊 详细监控: 每5个epoch显示三数据集F1/Loss")
        print(f"  预估总时间: 40-120分钟 (详细监控会稍慢)")
    else:
        print(f"  ⏰ 标准监控: 只在训练完成后显示三数据集F1/Loss")
        print(f"  预估总时间: 30-90分钟")
    
    # 显示超参数预览
    print(f"\n🔧 超参数预览:")
    for config_name in configs_to_run:
        config = TABNET_COMPLETE_CONFIGS[config_name]
        print(f"  {config_name}: LR={config['learning_rate']:.2e}, "
              f"Batch={config['batch_size']}, Gamma={config['gamma']:.2f}, "
              f"Max_epochs={config['max_epochs']}")
    
    # 执行完整实验
    final_model, all_candidates = experiment_manager.run_complete_experiment(
        configs_to_run=configs_to_run,
        detailed_monitoring=detailed_monitoring
    )
    
    # 生成所有报告和可视化
    if final_model and all_candidates:
        report_paths = experiment_manager.generate_all_reports_and_visualizations(
            final_model, all_candidates
        )
        
        # 最终总结
        print("\n" + "🎉"*20)
        print("实验完成 - 最终总结")
        print("🎉"*20)
        
        print(f"\n🏆 最终选择的模型: {final_model['config_name']}") 
        print(f"📊 关键性能指标:")
        print(f"  ├── 验证F1: {final_model['val_f1_macro']:.4f}")
        print(f"  ├── 测试F1: {final_model['test_f1_macro']:.4f}")
        print(f"  ├── 验证准确率: {final_model['val_accuracy']:.4f}")
        print(f"  ├── 测试准确率: {final_model['test_accuracy']:.4f}")
        print(f"  ├── 过拟合Gap: {final_model['overfitting_gap']:+.4f}")
        print(f"  ├── 泛化Gap: {final_model['generalization_gap']:+.4f}")
        
        best_config = TABNET_COMPLETE_CONFIGS[final_model['config_name']]
        print(f"  └── 关键参数: LR={best_config['learning_rate']:.2e}, "
              f"Gamma={best_config['gamma']:.2f}, "
              f"Batch={best_config['batch_size']}")

        print(f"\n📁 输出文件:")
        print(f"  ├── 📋 对比报告: {report_paths['report_path']}")
        print(f"  ├── 📊 完整可视化: {report_paths['visualization_path']}")
        print(f"  └── 📝 实验总结: {report_paths['summary_path']}")
        
        # 与4x4096全连接网络对比
        print(f"\n🔄 与4×4096全连接网络对比:")
        print(f"  ├── TabNet参数量: ~{TABNET_COMPLETE_CONFIGS[final_model['config_name']]['estimated_params']}")
        print(f"  ├── 全连接网络参数量: ~35M")
        print(f"  ├── TabNet测试F1: {final_model['test_f1_macro']:.4f}")
        print(f"  ├── 可解释性: ✅ (特征重要性可用)")
        print(f"  └── 过拟合控制: {'✅' if final_model['overfitting_gap'] < 0.05 else '⚠️'}")
        
        # 实验成功评估
        evaluate_experiment_success(final_model, all_candidates)
        
        return final_model, all_candidates, experiment_manager
    
    else:
        print("❌ 实验失败，没有成功训练的模型")
        return None, None, experiment_manager

def evaluate_experiment_success(final_model, all_candidates):
    """评估实验成功程度"""
    
    print(f"\n🎯 实验成功评估:")
    
    val_f1 = final_model['val_f1_macro']
    test_f1 = final_model['test_f1_macro']
    overfitting_gap = final_model['overfitting_gap']
    generalization_gap = abs(final_model['generalization_gap'])
    
    success_score = 0
    success_reasons = []
    
    # 基础成功标准
    if val_f1 >= 0.75:
        success_score += 25
        success_reasons.append(f"验证F1达到75%: {val_f1:.3f}")
    
    if test_f1 >= 0.70:
        success_score += 25
        success_reasons.append(f"测试F1达到70%: {test_f1:.3f}")
    
    # 过拟合控制
    if overfitting_gap < 0.05:
        success_score += 20
        success_reasons.append(f"过拟合控制良好: {overfitting_gap:+.3f}")
    elif overfitting_gap < 0.10:
        success_score += 10
        success_reasons.append(f"过拟合适中: {overfitting_gap:+.3f}")
    
    # 泛化能力
    if generalization_gap < 0.03:
        success_score += 20
        success_reasons.append(f"泛化能力优秀: {generalization_gap:.3f}")
    elif generalization_gap < 0.05:
        success_score += 10
        success_reasons.append(f"泛化能力良好: {generalization_gap:.3f}")
    
    # 模型多样性
    val_f1_range = max([c['val_f1_macro'] for c in all_candidates]) - min([c['val_f1_macro'] for c in all_candidates])
    if val_f1_range > 0.05:
        success_score += 10
        success_reasons.append(f"模型多样性好: F1范围{val_f1_range:.3f}")
    
    # 评估等级
    if success_score >= 90:
        level = "🏆 优秀"
        color = "绿色"
    elif success_score >= 70:
        level = "✅ 良好"
        color = "蓝色"
    elif success_score >= 50:
        level = "⚠️ 一般"
        color = "黄色"
    else:
        level = "❌ 需改进"
        color = "红色"
    
    print(f"  📊 综合评分: {success_score}/100 ({level})")
    print(f"  📋 成功原因:")
    for reason in success_reasons:
        print(f"    ├── {reason}")
    
    if success_score < 70:
        print(f"  💡 改进建议:")
        if val_f1 < 0.75:
            print(f"    ├── 考虑增加模型容量或调整超参数")
        if overfitting_gap > 0.05:
            print(f"    ├── 增强正则化或减少模型复杂度")
        if generalization_gap > 0.05:
            print(f"    ├── 检查验证集和测试集分布一致性")

# 🧪 快速测试模式
def quick_test_experiment():
    """快速测试模式 - 用于验证代码正确性"""
    
    print("\n🧪 快速测试模式")
    print("=" * 40)
    print("📋 测试内容:")
    print("  ├── 只运行micro_config和small_config")
    print("  ├── 减少最大训练轮数到50")
    print("  ├── 早停patience减少到15")
    print("  └── 预计时间: 10-20分钟")
    
    # 创建快速实验管理器
    quick_manager = SimplifiedExperimentManager(
        X_train=X_train_scaled,
        y_train=y_train_int,
        X_val=X_val_scaled,
        y_val=y_val_int,
        X_test=X_test_scaled,
        y_test=y_test_int
    )
    
    # 临时修改配置以加速测试
    original_configs = {}
    quick_configs = ['micro_config', 'small_config']
    
    for config_name in quick_configs:
        # 备份原始配置
        original_configs[config_name] = TABNET_COMPLETE_CONFIGS[config_name].copy()
        # 修改为快速测试配置
        TABNET_COMPLETE_CONFIGS[config_name]['max_epochs'] = 50
        TABNET_COMPLETE_CONFIGS[config_name]['patience'] = 15
    
    print(f"\n🚀 开始快速测试...")
    
    start_time = time.time()
    
    try:
        # 运行快速实验
        final_model, candidates = quick_manager.run_complete_experiment(
            configs_to_run=quick_configs
        )
        
        # 生成简化报告
        if final_model and candidates:
            quick_manager.generate_all_reports_and_visualizations(final_model, candidates)
            
            test_time = time.time() - start_time
            
            print(f"\n✅ 快速测试完成!")
            print(f"⏱️ 用时: {test_time:.2f}秒 ({test_time/60:.1f}分钟)")
            print(f"🏆 最佳模型: {final_model['config_name']}")
            print(f"📊 验证F1: {final_model['val_f1_macro']:.4f}")
            print(f"📊 测试F1: {final_model['test_f1_macro']:.4f}")
            
            return True
        else:
            print("❌ 快速测试失败")
            return False
            
    except Exception as e:
        print(f"❌ 快速测试出错: {e}")
        return False
    
    finally:
        # 恢复原始配置
        for config_name in quick_configs:
            if config_name in original_configs:
                TABNET_COMPLETE_CONFIGS[config_name] = original_configs[config_name]

# 🎯 主执行入口
if __name__ == "__main__":
    
    print("="*80)
    print("🚀 TabNet 5层级配置简化实验")
    print("="*80)
    print("📋 实验特色:")
    print("  ├── 5个不同规模的TabNet配置 (Micro → XLarge)")
    print("  ├── 精心调优的预定义超参数")
    print("  ├── 完整训练监控 (每epoch三数据集F1/Loss)")
    print("  ├── 基于验证F1的Early Stopping")
    print("  ├── 最终模型选择和完整分析报告")
    print("  ├── 实时可视化和性能分析")
    print("  └── 与4×4096全连接网络参数量对齐")
    
    print(f"\n🔧 预定义超参数亮点:")
    print(f"  ├── 小模型: 高学习率(8e-3) + 强正则化(gamma=1.8)")
    print(f"  ├── 中等模型: 平衡参数(lr=3e-3, gamma=1.4)")
    print(f"  ├── 大模型: 保守训练(lr=2e-3, gamma=1.25)")
    print(f"  └── 超大模型: 精细调优(lr=1.5e-3, gamma=1.15)")
    
    # 提供选择菜单
    print(f"\n🎮 选择运行模式:")
    print(f"  1. 标准实验 (所有5个配置, 训练完成后显示F1)")
    print(f"  2. 详细监控实验 (所有5个配置, 每5个epoch显示三数据集F1)")
    print(f"  3. 快速测试 (2个配置, 标准监控)")
    print(f"  4. 详细快速测试 (2个配置, 详细监控)")
    print(f"  5. 自定义配置")
    
    choice = input("\n请选择 (1/2/3/4/5): ").strip()
    
    if choice == '1':
        print("\n🚀 启动标准实验...")
        total_start_time = time.time()
        
        try:
            final_model, all_candidates, experiment_manager = main_simplified_experiment(detailed_monitoring=False)
            
            total_time = time.time() - total_start_time
            
            if final_model and all_candidates:
                print(f"\n" + "🎉"*30)
                print("🎊 标准实验圆满完成! 🎊")
                print("🎉"*30)
                
                print(f"\n⏱️ 总耗时: {total_time:.2f}秒 ({total_time/60:.1f}分钟)")
                print(f"📁 所有结果已保存到: {export_path}")
                print(f"\n✅ 实验成功完成! 检查 {export_path} 目录获取所有结果。")
                
            else:
                print(f"\n❌ 实验失败，没有成功的模型")
                
        except Exception as e:
            print(f"\n💥 实验过程中发生错误: {e}")
            import traceback
            traceback.print_exc()
    
    elif choice == '2':
        print("\n🚀 启动检查点模式实验...")
        total_start_time = time.time()
        
        try:
            experiment_manager = SimplifiedExperimentManager(
                X_train=X_train_scaled,
                y_train=y_train_int,
                X_val=X_val_scaled,
                y_val=y_val_int,
                X_test=X_test_scaled,
                y_test=y_test_int
            )
            
            # 运行实验，每5个epoch保存检查点
            configs_to_run = ['xlarge_config', 'large_config', 'medium_config']
            experiment_manager.run_complete_experiment(
                configs_to_run=configs_to_run,
                checkpoint_frequency=5  # 每5个epoch保存一次
            )
            
            total_time = time.time() - total_start_time
            
            print(f"\n✅ 检查点模式实验完成!")
            print(f"⏱️ 总耗时: {total_time:.2f}秒 ({total_time/60:.1f}分钟)")
            print(f"📁 所有结果已保存到: {export_path}")
            print(f"💾 检查点保存在: {os.path.join(export_path, 'checkpoints')}")
            print(f"\n可以使用CheckpointAnalyzer进行详细的过拟合分析")
            
        except Exception as e:
            print(f"\n💥 实验过程中发生错误: {e}")
            import traceback
            traceback.print_exc()
        
    elif choice == '3':
        print("\n🧪 启动快速测试...")
        success = quick_test_experiment()
        if success:
            print(f"\n✅ 快速测试成功! 可以运行完整实验了。")
        else:
            print(f"\n❌ 快速测试失败，请检查代码和环境。")
    
    elif choice == '4':
        print("\n🧪🔍 启动详细监控快速测试...")
        
        # 临时修改quick_test_experiment支持详细监控
        quick_manager = SimplifiedExperimentManager(
            X_train=X_train_scaled,
            y_train=y_train_int,
            X_val=X_val_scaled,
            y_val=y_val_int,
            X_test=X_test_scaled,
            y_test=y_test_int
        )
        
        # 临时修改配置以加速测试
        original_configs = {}
        quick_configs = ['micro_config', 'small_config']
        
        for config_name in quick_configs:
            # 备份原始配置
            original_configs[config_name] = TABNET_COMPLETE_CONFIGS[config_name].copy()
            # 修改为快速测试配置
            TABNET_COMPLETE_CONFIGS[config_name]['max_epochs'] = 30
            TABNET_COMPLETE_CONFIGS[config_name]['patience'] = 10
        
        start_time = time.time()
        
        try:
            # 运行详细监控快速实验
            final_model, candidates = quick_manager.run_complete_experiment(
                configs_to_run=quick_configs,
                detailed_monitoring=True
            )
            
            if final_model and candidates:
                quick_manager.generate_all_reports_and_visualizations(final_model, candidates)
                
                test_time = time.time() - start_time
                
                print(f"\n✅ 详细监控快速测试完成!")
                print(f"⏱️ 用时: {test_time:.2f}秒 ({test_time/60:.1f}分钟)")
                print(f"🏆 最佳模型: {final_model['config_name']}")
                print(f"📊 验证F1: {final_model['val_f1_macro']:.4f}")
                print(f"📊 测试F1: {final_model['test_f1_macro']:.4f}")
                print(f"📊 详细的epoch监控已记录")
            else:
                print("❌ 详细监控快速测试失败")
                
        except Exception as e:
            print(f"❌ 详细监控快速测试出错: {e}")
        
        finally:
            # 恢复原始配置
            for config_name in quick_configs:
                if config_name in original_configs:
                    TABNET_COMPLETE_CONFIGS[config_name] = original_configs[config_name]
    
    elif choice == '5':
        print("\n⚙️ 自定义配置模式")
        print("可选配置:", list(TABNET_COMPLETE_CONFIGS.keys()))
        custom_configs = input("请输入配置名称 (用逗号分隔): ").strip().split(',')
        custom_configs = [c.strip() for c in custom_configs if c.strip() in TABNET_COMPLETE_CONFIGS]
        
        if custom_configs:
            monitor_choice = input("是否启用详细监控? (y/n): ").strip().lower()
            detailed = monitor_choice in ['y', 'yes', '是']
            
            print(f"\n🚀 启动自定义实验: {custom_configs}")
            print(f"📊 详细监控: {'启用' if detailed else '关闭'}")
            
            experiment_manager = SimplifiedExperimentManager(
                X_train=X_train_scaled,
                y_train=y_train_int,
                X_val=X_val_scaled,
                y_val=y_val_int,
                X_test=X_test_scaled,
                y_test=y_test_int
            )
            
            final_model, all_candidates = experiment_manager.run_complete_experiment(
                configs_to_run=custom_configs,
                detailed_monitoring=detailed
            )
            
            if final_model and all_candidates:
                experiment_manager.generate_all_reports_and_visualizations(final_model, all_candidates)
                print(f"\n✅ 自定义实验完成!")
            else:
                print(f"\n❌ 自定义实验失败")
        else:
            print(f"\n❌ 无效的配置选择")
    
    else:
        print("\n❌ 无效选择，实验取消")



In [ ]:
# ===== TabNet 训练模型评估单元格 =====
# 此单元格用于加载已训练的TabNet模型并计算三个数据集的完整性能指标

import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, balanced_accuracy_score
from pytorch_tabnet.tab_model import TabNetClassifier
import os
import json
from datetime import datetime
import pandas as pd

def calculate_cross_entropy_loss(y_true, y_pred_proba, epsilon=1e-15):
    """计算交叉熵损失"""
    # 确保概率值在有效范围内
    y_pred_proba = np.clip(y_pred_proba, epsilon, 1 - epsilon)
    
    # 确保y_true是整数索引
    y_true = y_true.astype(int)
    
    # 计算交叉熵
    n_samples = len(y_true)
    ce_loss = -np.sum(np.log(y_pred_proba[np.arange(n_samples), y_true])) / n_samples
    
    return ce_loss

def evaluate_tabnet_model(model_path, X_train, y_train, X_val, y_val, X_test, y_test, 
                         config_name="xlarge_config", verbose=True):
    """
    评估已训练的TabNet模型
    
    参数:
        model_path: 模型文件路径
        X_train, y_train: 训练数据和标签
        X_val, y_val: 验证数据和标签
        X_test, y_test: 测试数据和标签
        config_name: 配置名称
        verbose: 是否打印详细信息
    
    返回:
        results: 包含所有评估指标的字典
    """
    
    print(f"\n{'='*60}")
    print(f"📊 {config_name} 模型评估")
    print(f"{'='*60}")
    
    try:
        # 1. 加载模型
        print(f"📂 加载模型: {model_path}")
        model = TabNetClassifier()
        model.load_model(model_path)
        print("✅ 模型加载成功")
        
        # 2. 检查模型输出维度
        print(f"\n🔍 模型检查:")
        # 做一个小批量预测来检查输出维度
        test_batch = X_train[:100]
        test_proba = model.predict_proba(test_batch)
        print(f"  - 模型输出类别数: {test_proba.shape[1]}")
        print(f"  - 数据集类别数: {len(np.unique(y_train))}")
        print(f"  - 标签范围: {y_train.min()} - {y_train.max()}")
        
        # 3. 预测所有数据集
        print(f"\n🚀 开始预测三个数据集...")
        
        # 训练集预测
        print("  - 预测训练集...")
        train_preds = model.predict(X_train)
        train_proba = model.predict_proba(X_train)
        
        # 验证集预测
        print("  - 预测验证集...")
        val_preds = model.predict(X_val)
        val_proba = model.predict_proba(X_val)
        
        # 测试集预测
        print("  - 预测测试集...")
        test_preds = model.predict(X_test)
        test_proba = model.predict_proba(X_test)
        
        print("✅ 预测完成")
        
        # 4. 处理可能的类别不匹配问题
        # 如果模型输出102类但数据只有101类，需要处理
        n_classes_model = train_proba.shape[1]
        n_classes_data = len(np.unique(y_train))
        
        if n_classes_model != n_classes_data:
            print(f"\n⚠️ 类别数不匹配: 模型输出{n_classes_model}类, 数据有{n_classes_data}类")
            print("  正在修正预测结果...")
            
            # 确保预测标签在有效范围内
            train_preds = np.clip(train_preds, 0, n_classes_data - 1)
            val_preds = np.clip(val_preds, 0, n_classes_data - 1)
            test_preds = np.clip(test_preds, 0, n_classes_data - 1)
            
            # 如果需要，截断概率矩阵
            if n_classes_model > n_classes_data:
                train_proba = train_proba[:, :n_classes_data]
                val_proba = val_proba[:, :n_classes_data]
                test_proba = test_proba[:, :n_classes_data]
                # 重新归一化
                train_proba = train_proba / train_proba.sum(axis=1, keepdims=True)
                val_proba = val_proba / val_proba.sum(axis=1, keepdims=True)
                test_proba = test_proba / test_proba.sum(axis=1, keepdims=True)
        
        # 5. 计算所有指标
        print(f"\n📈 计算性能指标...")
        
        results = {
            'config_name': config_name,
            'model_path': model_path,
            'evaluation_time': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            
            # F1 Score (Macro)
            'train_f1_macro': f1_score(y_train, train_preds, average='macro'),
            'val_f1_macro': f1_score(y_val, val_preds, average='macro'),
            'test_f1_macro': f1_score(y_test, test_preds, average='macro'),
            
            # Accuracy
            'train_accuracy': accuracy_score(y_train, train_preds),
            'val_accuracy': accuracy_score(y_val, val_preds),
            'test_accuracy': accuracy_score(y_test, test_preds),
            
            # Balanced Accuracy
            'train_balanced_acc': balanced_accuracy_score(y_train, train_preds),
            'val_balanced_acc': balanced_accuracy_score(y_val, val_preds),
            'test_balanced_acc': balanced_accuracy_score(y_test, test_preds),
            
            # Cohen's Kappa
            'train_kappa': cohen_kappa_score(y_train, train_preds),
            'val_kappa': cohen_kappa_score(y_val, val_preds),
            'test_kappa': cohen_kappa_score(y_test, test_preds),
            
            # Cross Entropy Loss
            'train_loss': calculate_cross_entropy_loss(y_train, train_proba),
            'val_loss': calculate_cross_entropy_loss(y_val, val_proba),
            'test_loss': calculate_cross_entropy_loss(y_test, test_proba),
            
            # 额外信息
            'n_train_samples': len(y_train),
            'n_val_samples': len(y_val),
            'n_test_samples': len(y_test),
            'n_classes_model': n_classes_model,
            'n_classes_data': n_classes_data,
        }
        
        # 计算Gap指标
        results['overfitting_gap'] = results['train_f1_macro'] - results['val_f1_macro']
        results['generalization_gap'] = results['val_f1_macro'] - results['test_f1_macro']
        
        # 6. 打印结果
        if verbose:
            print(f"\n{'='*60}")
            print(f"📊 {config_name} 评估结果")
            print(f"{'='*60}")
            
            print(f"\n📈 Macro F1 Score:")
            print(f"  🔹 训练集: {results['train_f1_macro']:.4f}")
            print(f"  🔹 验证集: {results['val_f1_macro']:.4f}")
            print(f"  🔹 测试集: {results['test_f1_macro']:.4f}")
            
            print(f"\n🎯 Accuracy:")
            print(f"  🔹 训练集: {results['train_accuracy']:.4f}")
            print(f"  🔹 验证集: {results['val_accuracy']:.4f}")
            print(f"  🔹 测试集: {results['test_accuracy']:.4f}")
            
            print(f"\n⚖️ Balanced Accuracy:")
            print(f"  🔹 训练集: {results['train_balanced_acc']:.4f}")
            print(f"  🔹 验证集: {results['val_balanced_acc']:.4f}")
            print(f"  🔹 测试集: {results['test_balanced_acc']:.4f}")
            
            print(f"\n📉 Cross Entropy Loss:")
            print(f"  🔹 训练集: {results['train_loss']:.4f}")
            print(f"  🔹 验证集: {results['val_loss']:.4f}")
            print(f"  🔹 测试集: {results['test_loss']:.4f}")
            
            print(f"\n🔍 模型分析:")
            print(f"  🔹 过拟合Gap (Train-Val F1): {results['overfitting_gap']:+.4f}")
            print(f"  🔹 泛化Gap (Val-Test F1): {results['generalization_gap']:+.4f}")
            
            # 性能评估
            if results['overfitting_gap'] < 0.02:
                overfit_status = "✅ 无过拟合"
            elif results['overfitting_gap'] < 0.05:
                overfit_status = "⚠️ 轻微过拟合"
            else:
                overfit_status = "❌ 明显过拟合"
            
            if abs(results['generalization_gap']) < 0.02:
                general_status = "✅ 泛化良好"
            elif abs(results['generalization_gap']) < 0.05:
                general_status = "⚠️ 泛化一般"
            else:
                general_status = "❌ 泛化较差"
            
            print(f"  🔹 过拟合状态: {overfit_status}")
            print(f"  🔹 泛化状态: {general_status}")
        
        # 7. 保存评估结果
        save_path = os.path.join(os.path.dirname(model_path), f'{config_name}_evaluation_results.json')
        with open(save_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"\n💾 评估结果已保存: {save_path}")
        
        return results
        
    except Exception as e:
        print(f"\n❌ 评估失败: {e}")
        import traceback
        traceback.print_exc()
        return None

def create_evaluation_summary(results_list, save_path=None):
    """创建评估结果汇总表"""
    
    if not results_list:
        print("❌ 没有评估结果")
        return
    
    # 创建DataFrame
    df = pd.DataFrame(results_list)
    
    # 选择关键列
    key_columns = [
        'config_name', 
        'train_f1_macro', 'val_f1_macro', 'test_f1_macro',
        'train_accuracy', 'val_accuracy', 'test_accuracy',
        'train_loss', 'val_loss', 'test_loss',
        'overfitting_gap', 'generalization_gap'
    ]
    
    df_summary = df[key_columns].round(4)
    
    print("\n📊 评估结果汇总表:")
    print("="*100)
    print(df_summary.to_string(index=False))
    
    if save_path:
        df_summary.to_csv(save_path, index=False)
        print(f"\n💾 汇总表已保存: {save_path}")
    
    return df_summary

# ===== 主评估代码 =====
if __name__ == "__main__":
    
    # 评估已训练的xlarge模型
    model_path = "./tabnet_simplified_experiment/models/xlarge_config_final_model.zip"
    
    # 确保数据已加载（使用之前notebook中的变量）
    print(f"📊 数据确认:")
    print(f"  训练集: {X_train_scaled.shape}, 标签: {y_train_int.shape}")
    print(f"  验证集: {X_val_scaled.shape}, 标签: {y_val_int.shape}")
    print(f"  测试集: {X_test_scaled.shape}, 标签: {y_test_int.shape}")
    
    # 执行评估
    results = evaluate_tabnet_model(
        model_path=model_path,
        X_train=X_train_scaled,
        y_train=y_train_int,
        X_val=X_val_scaled,
        y_val=y_val_int,
        X_test=X_test_scaled,
        y_test=y_test_int,
        config_name="xlarge_config",
        verbose=True
    )
    
    # 如果有多个模型，可以批量评估
    # model_configs = ['xlarge_config', 'large_config', 'medium_config']
    # results_list = []
    # for config in model_configs:
    #     model_path = f"./tabnet_simplified_experiment/models/{config}_final_model.zip"
    #     if os.path.exists(model_path):
    #         results = evaluate_tabnet_model(...)
    #         if results:
    #             results_list.append(results)
    # 
    # # 创建汇总表
    # create_evaluation_summary(results_list, "./evaluation_summary.csv")